# SCADA Functionality Test Visualiser

**Author:** Erick Chauke

SCADA stands for Supervisory Control and Data Acquisition, the system that runs a power plant and logs its measurements every second. Each grid-code functionality test leaves one of these logger spreadsheets, and today every one is checked by eye against the acceptance procedure. This notebook starts to automate that. Drop a test export into `data/`, run the notebook, and it parses the channels, compares the measured values against their setpoints and control modes, and plots whether the plant met each part of the procedure. The procedure is the SCADA Functionality Test Record (Rev 3) from NCSS, the National Control System Support group of the grid operator [1]. It is built to be general, so nothing is tied to one site. The single config cell below is all that changes between tests, and new sections are added one at a time. The first is the curtailment test.

## Setup

The config cell below is the only thing you edit. To run a new test, drop its spreadsheet into `data/` and run all cells. It locates the workbook, sets the site, time zone and highlighted event windows, and creates `outputs/`. Each figure is saved there with a short form of the site name as a prefix, so results from different plants never overwrite one another.

## Config parameters

You edit only the config cell. The parameters below are listed in the order you normally set them; usually only the first few change from one test to the next. Each grid-code threshold carries its source in a comment beside it in the cell.

| Parameter | What it does | Example value |
|---|---|---|
| `SITE_NAME` | Set this first. Plant name shown in every figure title, and the short slug that prefixes saved figures so different plants never overwrite each other. | `"Hartebeesthoek"` |
| `TIME_ZONE_LABEL` | The time zone the logger timestamps are recorded in; it labels the time axes. | `"UTC"` |
| `INPUT_GLOB` | Filename pattern that selects the workbook in `data/`. Narrow it only if more than one spreadsheet is present. | `"*.xlsx"` or `"Hartebeesthoek*.xlsx"` |
| `COLUMN_OVERRIDES` | Pins a role to an exact column name, used only when a channel is named so unusually it is not matched automatically. | `{"poc_p": "Active Power MW"}` |
| Grid-code thresholds and reading aids | The grid-code limits (over-frequency and trip points, active-power accuracy and timing, voltage accuracy and timing, AGC movement) plus the notebook reading aids each test uses (the curtailment, ramp, frequency and stop acceptance tolerances and read windows). Change the grid-code values only if a different grid code applies. | `F4_OVER_FREQUENCY_HZ = 50.5` |
| `EVENT_WINDOWS` | Restricts one test to part of the day; leave an entry `None` to scan the whole record. | `{"curtailment": ("2026-05-27 08:10", "2026-05-27 08:40")}` |


In [1]:
# Single config cell. To point this notebook at a different test, drop the new
# spreadsheet into the data/ folder and (only if more than one file is present)
# adjust INPUT_GLOB below. Nothing else in the notebook is edited to swap sites.

import re                 # regular expressions, used only to turn the site name into a safe file-name slug
from pathlib import Path  # tidy, cross-platform file paths for the data/ and outputs/ folders

DATA_DIR = Path("data")          # input files live here (gitignored)
# Filename pattern that selects the logger workbook inside data/. The "*" stands for any
# run of characters, so "*.xlsx" matches every Excel file in the folder. With a single
# workbook present that is all you need. If several are present, make the pattern
# specific enough to match only the one you want. For example "Hartebeesthoek.xlsx"
# matches that one exact file, and "Hartebeesthoek*.xlsx" matches any file whose name
# starts with the site name.
INPUT_GLOB = "*.xlsx"
OUTPUT_DIR = Path("outputs")     # every figure is saved here

SITE_NAME = "Hartebeesthoek"     # plant name shown in titles
TIME_ZONE_LABEL = "UTC"         # the logger timestamps are in this zone

# Optional scope restriction per test. Each section finds every occurrence of its test on
# its own, so leave an entry as None to scan the whole record (the normal case). Set a
# (start, end) pair only if you want to limit a test to one part of the day. The ceiling,
# ramp rates and exact times are always read from the data, never hard-coded.
EVENT_WINDOWS = {
    "curtailment": None,
    "power_gradient": None,
    "frequency": None,
    "delta": None,
    "voltage": None,
    "reactive_power": None,
    "power_factor": None,
    "stop_start": None,
}

# Curtailment and power gradient acceptance tolerances. These are notebook reading aids,
# not grid-code values: the NCSS record (Sections 7 and 9, p.10) asks the plant to curtail
# "to set point", return to "full output" and ramp "at the specified ramp rate" without
# giving numbers, so the small margins below are practical choices for calling a pass. The
# curtailment checks allow whichever is larger of the MW margin or the fraction.
CURTAIL_CEILING_TOLERANCE_MW = 1.0         # smallest margin allowed above the ceiling, in MW
CURTAIL_CEILING_TOLERANCE_FRACTION = 0.05  # margin allowed above the ceiling as a share of it (5 percent)
CURTAIL_RETURN_TOLERANCE_MW = 2.0          # largest shortfall allowed below the earlier level, in MW
CURTAIL_RETURN_TOLERANCE_FRACTION = 0.10   # shortfall allowed below the earlier level as a share of it (10 percent)
CURTAIL_SENT_TOGETHER_SECONDS = 5          # setpoint and mode within this many seconds count as one step
RAMP_RISE_LOW_FRACTION = 0.1               # measure the ramp rate over the steady middle of the move, from 10 percent risen ...
RAMP_RISE_HIGH_FRACTION = 0.9              # ... to 90 percent risen, skipping the dead time at the start and the flattening at the end
RAMP_MIN_MOVE_MW = 0.5                     # a setpoint move smaller than this is too small to count as a ramp
RAMP_MATCH_TOLERANCE_FRACTION = 0.2        # measured rate within this share of the commanded limit reads as "close to the limit"

# Frequency response thresholds. These are the only frequency values supplied by hand,
# because the test sheet does not record them. They come from the South African Grid Code
# requirements for renewable power plants [2]: above the over-frequency point the plant
# must start reducing power, and above the trip point held for the trip hold time it must
# disconnect. The droop is read from the sheet, not set here (the code lets it be set
# anywhere from 0 to 10 percent, agreed with the system operator, SAGC 6.2(5)), and the
# nominal frequency is read from the measured grid frequency.
F4_OVER_FREQUENCY_HZ = 50.5   # when grid frequency rises above this, the plant must start cutting active power; SAGC 6.1(2) and Figure 6, p.19
F5_TRIP_HZ = 51.5             # above this frequency the plant must disconnect from the grid (a trip); SAGC 6.1(3), p.19
TRIP_HOLD_SECONDS = 4         # how long the frequency must stay above the trip point before the plant must disconnect; SAGC 6.1(3), p.19
# The following are notebook reading aids, not grid-code figures. They set what the
# notebook treats as a clear response, a trip and a recovery when reading the measured output.
FREQ_RESPONSE_MIN_DROP_MW = 2.0        # a drop of at least this many MW counts as a clear frequency response ...
FREQ_RESPONSE_MIN_DROP_FRACTION = 0.1  # ... or this fraction of the reference power, whichever is larger
FREQ_TRIP_OUTPUT_PERCENT = 5.0         # output at or below this percent of reference reads as the plant having tripped (disconnected)
FREQ_RECOVERY_FRACTION = 0.9           # output back to at least this share of its reference counts as recovered
FREQ_REFERENCE_LOOKBACK_SECONDS = 30   # how far back to read the reference output just before the frequency crossing
FREQ_RECOVERY_MARK_GAP_SECONDS = 10    # mark recovery as its own moment only if it is at least this long after the drop back below the point

# Delta production constraint. Delta is the reserve the plant holds back, as a percentage of
# its available power, and the delivered reduction is judged against the active-power accuracy
# below. The read windows are notebook reading aids for measuring the available power and the
# settled output from this logger, which has no available-power channel. The SAGC commence and
# complete timing and the 3 percent minimum-capability requirements are noted in the delta
# section but not assessed by this notebook.
ACTIVE_POWER_ACCURACY_PERCENT = 2.0   # how closely a commanded active-power change must be met; the grid code active-power control accuracy that the delta and frequency Figure 6 shape checks are judged against; SAGC 6.2(11), p.21 (the code allows the larger of this or 0.5% of rated power)
DELTA_AVAILABLE_LOOKBACK_SECONDS = 30  # how far back to read the available power just before delta mode comes on
DELTA_SETTLED_TAIL_SECONDS = 20        # how much of the end of the on-window to read the settled output over
DELTA_RECOVERY_SAMPLES = 20            # how many samples after mode off to read the recovered output over

# Voltage mode. Voltage control holds the point of connection voltage at a reference; the
# reference voltage and the droop are read from the sheet. The grid code accuracy is set
# here; the threshold and read window below are notebook reading aids for judging the hold
# and the reactive-power direction from the data. The SAGC commence and complete timing
# requirements are noted in the voltage section but not assessed by this notebook.
VOLTAGE_ACCURACY_PERCENT = 0.5         # the held voltage must stay within this percent of the nominal voltage; SAGC 8.3(3), p.28
VOLTAGE_HELD_FRACTION = 0.9            # the voltage must sit inside the accuracy band for at least this share of the window to count as held
VOLTAGE_REACTIVE_DEADBAND_MVAR = 0.5   # reactive power between plus and minus this counts as near zero, not injected or absorbed
VOLTAGE_STEP_REACTIVE_SECONDS = 60     # how long after a setpoint step to read the reactive-power direction

# Reactive power (Q) and power factor (PF) modes. The measurement must track its commanded
# setpoint within the accuracy below. The settle tail is how much of the end of each
# commanded level is read so the step into it is not counted. The power-factor plot break
# lifts the pen across the +1 to -1 sign flip at unity, which is the same physical point and
# would otherwise be drawn as a false vertical line.
REACTIVE_POWER_ACCURACY_MVAR = 2.0     # measured reactive power must track its setpoint within this many MVAr (reading aid; the grid code states reactive accuracy as a share of rated MVAr)
POWER_FACTOR_ACCURACY = 0.02           # measured power factor must track its setpoint within this; the procedure quotes +/-0.02
REACTIVE_SETTLE_TAIL_SECONDS = 20      # read the settled reactive power or power factor over this much of the end of each commanded level
PF_PLOT_BREAK = 1.0                    # a jump larger than this between samples is a sign flip at unity, so the line is broken there rather than drawn across it
PF_UNITY_REACTIVE_MVAR = 2.0           # when reactive power is within this of zero the plant is at unity, where the power-factor sign is arbitrary, so the measured trace is drawn at +1

# Stop and start: the NCSS test (Section 13, p.12) asks only that the plant ramps down to
# zero and back up. "Zero" on a live feed is never exactly zero, so the notebook counts the
# plant as stopped when its active power falls below this small fraction of its own running
# maximum (read from the data, not assumed). This fraction is a practical notebook choice,
# not a grid-code figure.
STOP_FRACTION = 0.05               # at or below this share of the plant's own running maximum, it counts as stopped
STOP_LEVEL_BEFORE_SECONDS = 90     # how far back to read the running level just before a stop (notebook reading aid)
STOP_LEVEL_AFTER_SECONDS = 120     # how far forward to read the running level just after a start (notebook reading aid)
STOP_RECOVERY_FRACTION = 0.8        # output back to at least this share of its pre-stop level counts as a successful restart (notebook reading aid)

# AGC signal verification: the NCSS procedure (Section 15, p.13) asks for the plant to be
# moved "atleast 20 or 30 MW" so the control response can be observed [1]; 20 is the smaller
# of the two and is used here as the minimum movement to look for.
AGC_MOVE_MW = 20                   # the smallest amount the plant must be moved so its control response can be seen; NCSS Section 15, p.13

# Escape hatch for odd spreadsheets. The notebook normally finds each channel on its
# own, but if it ever guesses wrong, map the role to the exact column name here and it
# takes priority. Roles: poc_p, sp_p, ap_mode, pg_mode, ramp_up, ramp_down, f_control,
# grid_freq, droop_f, delta_mode, delta_sp, v_mode, v_meas, v_sp, droop_v, q_mode,
# q_meas, q_sp, pf_mode, pf_meas, pf_sp, hi_limit, lo_limit, sentout, generated,
# agc_mode, sp_feedback, timestamp, date, time. Example:
#   COLUMN_OVERRIDES = {"poc_p": "Active Power MW"}
COLUMN_OVERRIDES = {}

# Resolve the single input workbook without hard-coding its (confidential) name.
_candidates = sorted(DATA_DIR.glob(INPUT_GLOB))
assert len(_candidates) >= 1, f"no file matching {INPUT_GLOB} found in {DATA_DIR}"
INPUT_FILE = _candidates[0]

# Namespace saved figures by a safe slug of the site name, never the raw (and
# confidential) file name, so outputs from different sites never overwrite one
# another and no committed figure leaks the source file name.
SITE_SLUG = re.sub(r"[^0-9a-zA-Z]+", "_", SITE_NAME).strip("_").lower()

OUTPUT_DIR.mkdir(exist_ok=True)
_scope = {k: v for k, v in EVENT_WINDOWS.items() if v}
print(f"Site: {SITE_NAME} | timezone: {TIME_ZONE_LABEL}")
print(f"Input workbook resolved from {DATA_DIR}/ ({len(_candidates)} match)")
print(f"Figures will be saved to {OUTPUT_DIR}/ with prefix '{SITE_SLUG}_'")
print(f"Test scope overrides: {_scope or 'none, scanning the whole record'}")
print(f"Column overrides: {COLUMN_OVERRIDES or 'none'}")

Site: Hartebeesthoek | timezone: UTC
Input workbook resolved from data/ (1 match)
Figures will be saved to outputs/ with prefix 'hartebeesthoek_'
Test scope overrides: none, scanning the whole record
Column overrides: none


## Expected input structure

This section is a reader's orientation, not a contract. The cleaning and channel resolution cell further down is the authoritative matcher: it finds each channel by pattern and, when you run it, prints the exact column it resolved for every role. Read that printed list to confirm what was matched in your sheet; read the table here to understand what the notebook is looking for.

**Workbook shape.** The logger export may hold several sheets; the notebook reads the largest data sheet. Expect roughly one row per second, each row a timestamped snapshot of the plant.

**Timestamp.** Either a single combined date-and-time column, or a separate `Date` and `Time` pair. Both are handled, including Excel serial dates and quoted time strings such as `'08:00:47'`.

**Three families of columns.** Measured channels (what the plant did), setpoints (what it was told to do), and on/off mode flags (which control mode was active). A column is treated as a mode flag only when its name looks like one (mode, flag, status, enable) and its values are binary, so a setpoint that happens to sit at a constant value is never mistaken for a flag.

**Roles the notebook resolves.** POC is the point of connection, where the plant meets the grid. MVAr is megavolt-amperes reactive, the unit of reactive power. AGC is Automatic Generation Control. The example column names below are illustrative only and are matched ignoring case, spacing and punctuation.

| Role | Meaning | Example column name |
|---|---|---|
| `poc_p`, `sp_p` | active power at the POC and its setpoint, in MW (megawatts) | "POC: P (MW)", "SP: P (MW)" |
| `ap_mode`, `pg_mode` | active-power (curtailment) and power-gradient mode flags | "Mode: Active Power", "Mode: Power Gradient" |
| `ramp_up`, `ramp_down` | commanded ramp rates | "Ramp Up", "Ramp Down" |
| `f_control`, `grid_freq`, `droop_f` | injected test frequency, measured grid frequency, frequency droop | "f used by f control", "POC: Freq (Hz)", "SP: Droop F (%)" |
| `delta_mode`, `delta_sp`, `available` | delta reserve mode, its setpoint and available power | "Mode: p-Delta", "SP: P-Delta", "P Available" |
| `v_mode`, `v_meas`, `v_sp`, `droop_v` | voltage mode, measured POC voltage, its setpoint and droop | "Mode: V", "POC: Voltage", "SP: Voltage", "Droop V" |
| `q_mode`, `q_meas`, `q_sp` | reactive-power mode, measured reactive power and its setpoint, in MVAr | "Mode: Q", "POC: Q (MVAr)", "SP: Q" |
| `pf_mode`, `pf_meas`, `pf_sp` | power-factor mode, measured power factor (PF) and its setpoint | "Mode: PF", "POC: PF", "SP: PF" |
| `agc_mode`, `hi_limit`, `lo_limit`, `sentout`, `generated`, `sp_feedback` | AGC telemetry, resolved if present and reported as not captured if absent | "AGC Status", "High Regulation Limit", "Sent Out", "Generated" |

Any channel named so unusually that it is not matched can be pinned by hand through `COLUMN_OVERRIDES` in the config cell, which always wins.


## Data ingestion and inspection

Before trusting any figure, this section loads the workbook and reports what is inside it: how many sheets there are, the row and column counts, and the type of each column. When a workbook has several sheets the largest one is taken as the logged time series. Sheet names are not printed, because they can carry identifiers that should stay confidential. The output here is a structural check rather than a plot, and it is the moment to confirm the data matches what the test was meant to capture.

In [2]:
import pandas as pd  # reads the Excel workbook and gives the table (DataFrame) tools used throughout

# Load every sheet so the real structure is confirmed before any narrative is built on
# it. sheet_name=None keeps a multi-sheet workbook from being reduced to its first tab,
# and na_values catches the string sentinels Excel exports leave behind.
sheets = pd.read_excel(INPUT_FILE, sheet_name=None, na_values=["NULL", "None", "NaN", ""])

print(f"Sheets found: {len(sheets)}")
for i, frame in enumerate(sheets.values(), start=1):
    print(f"  sheet {i}: {frame.shape[0]} rows x {frame.shape[1]} columns")

# Work from the sheet with the most rows, which is the logged time series. Sheet names
# are deliberately not printed, as they can carry confidential identifiers.
raw = max(sheets.values(), key=len)
print(f"\nUsing the largest sheet: {raw.shape[0]} rows x {raw.shape[1]} columns")

print("\nColumns and dtypes:")
for col, dtype in raw.dtypes.items():
    print(f"  {col:<26} {dtype}")

Sheets found: 1
  sheet 1: 7147 rows x 23 columns

Using the largest sheet: 7147 rows x 23 columns

Columns and dtypes:
  Date                       datetime64[us]
  Time UTC(NC2)              str
  POC: P (MW)                float64
  POC: Q (MVAr)              float64
  POC: Freq (Hz)             float64
  POC: PF                    float64
  POC: Average Voltage (KV)  float64
  SP: P (MW)                 int64
  SP: Q (MVAr)               int64
  SP:Voltage (kV)            int64
  SP: PF                     int64
  SP: P-Delta (%)            int64
  f used by f control (during test) float64
  SP:Ramp up (MW/min)        int64
  SP:Ramp down (MW/min)      int64
  Mode:Q                     int64
  Mode:V                     int64
  Mode:PF                    int64
  Mode: Active Power         int64
  Mode: p-Delta              int64
  Mode: Power Gradient       int64
  SP:Droop V (%)             int64
  SP:Droop F (%)             int64


## Data cleaning, parsing and channel resolution

Different sites do not label their columns identically, so the notebook does not assume fixed names. Each value it needs is described by a role, such as the measured active power, the active-power setpoint, the curtailment-mode flag, and the date and time. A small resolver then matches that role to whatever column carries it, ignoring case, spacing and punctuation. If a match is ever wrong, the real name can be forced through `COLUMN_OVERRIDES` in the config cell.

With the columns resolved, the cleaning runs. The time of day is read whether it arrives as a quoted string, a plain time, a full datetime or an Excel fraction of a day, then joined to the date to form the timestamp index. Columns that hold only zero and one become off and on flags, and every other channel is forced to a number so stray text turns into a missing value. The printout reports what was resolved and the time span, without showing any raw rows.

#### Channel resolver

Each value the notebook needs is found by its role, not its column name. This builds the list of name fragments per role and the `resolve()` function the whole notebook calls.


In [3]:
# --- Channel resolution -------------------------------------------------------
# Sites label columns differently, so each logical channel (its role) is matched to
# whatever column carries it, ignoring case, spacing and punctuation. If a guess is
# ever wrong, set the real name in COLUMN_OVERRIDES in the config cell and it wins.

def _norm(name):
    # Lower-case, then turn any run of non-alphanumeric characters into one space.
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", str(name).lower())).strip()

# Each role lists a few text fragments to look for in a column's cleaned name (lower-cased
# with punctuation turned into spaces). The resolver below takes the first column whose
# cleaned name matches any one of a role's fragments. The fragments are regular
# expressions, a small matching language: '.*' means 'any text in between', and a '\b' on
# each side of a short word like 'p' or 'q' pins it so it matches on its own and not inside
# a longer word. For example the fragment 'poc.*\bp\b.*mw' matches the column 'POC: P (MW)'.
CHANNEL_PATTERNS = {
    "timestamp": [r"date.*time", r"\btimestamp\b", r"\bdatetime\b"],
    "date":      [r"\bdate\b"],
    "time":      [r"\btime\b"],
    "poc_p":     [r"poc.*\bp\b.*mw", r"poc.*active power", r"active power.*poc",
                  r"measured.*\bp\b.*mw"],
    "sp_p":      [r"\bsp\b.*\bp\b.*mw", r"set ?point.*\bp\b.*mw", r"\bp\b.*set ?point"],
    "ap_mode":   [r"mode.*active power", r"active power.*mode", r"curtail.*mode",
                  r"mode.*curtail"],
    "pg_mode":   [r"mode.*power gradient", r"power gradient.*mode", r"mode.*gradient",
                  r"gradient.*mode"],
    "ramp_up":   [r"ramp up", r"up ramp", r"ramp.*up.*min"],
    "ramp_down": [r"ramp down", r"down ramp", r"ramp.*down.*min"],
    "f_control": [r"f used", r"f control"],
    "grid_freq": [r"poc.*freq", r"grid.*freq", r"measured.*freq"],
    "droop_f":   [r"droop f", r"droop.*\bf\b"],
    "delta_mode": [r"mode.*delta", r"delta.*mode"],
    "delta_sp":   [r"sp.*delta", r"\bp delta\b", r"set ?point.*delta"],
    "available":  [r"\bavail", r"p ?avail", r"available power"],
    "v_mode":     [r"mode.*\bv\b"],
    "q_mode":     [r"mode.*\bq\b"],
    "pf_mode":    [r"mode.*\bpf\b"],
    "v_meas":     [r"poc.*volt", r"average voltage"],
    "q_meas":     [r"poc.*\bq\b", r"poc.*mvar"],
    "pf_meas":    [r"poc.*\bpf\b", r"poc.*power factor"],
    "v_sp":       [r"sp.*volt", r"set ?point.*volt"],
    "q_sp":       [r"sp.*\bq\b", r"sp.*mvar"],
    "pf_sp":      [r"sp.*\bpf\b", r"sp.*power factor"],
    "droop_v":    [r"droop v", r"droop.*\bv\b"],
    "agc_mode":   [r"agc.*status", r"\bagc\b.*on", r"mode.*agc", r"agc.*mode"],
    "hi_limit":   [r"high.*regulat", r"regulat.*high", r"\bhl\b", r"high.*limit"],
    "lo_limit":   [r"low.*regulat", r"regulat.*low", r"\bll\b", r"low.*limit"],
    "sentout":    [r"sent ?out"],
    "generated":  [r"generated"],
    "sp_feedback": [r"set ?point feedback", r"sp.*feedback", r"\bfeedback\b"],
}

_norms = {col: _norm(col) for col in raw.columns}

def resolve(role, required=False):
    override = COLUMN_OVERRIDES.get(role)
    if override is not None:
        if override not in raw.columns:
            raise KeyError(f"COLUMN_OVERRIDES['{role}'] = '{override}' is not a column")
        return override
    for pattern in CHANNEL_PATTERNS.get(role, []):
        for col, norm in _norms.items():
            if re.search(pattern, norm):
                return col
    if required:
        raise KeyError(
            f"could not find a column for the '{role}' channel. Set "
            f"COLUMN_OVERRIDES['{role}'] in the config cell to one of: {list(raw.columns)}"
        )
    return None

# A short confirmation that this step ran (useful now the section is split into steps).
print(f"Channel resolver ready: {len(CHANNEL_PATTERNS)} roles, matching against {len(_norms)} columns.")


Channel resolver ready: 31 roles, matching against 23 columns.


#### Timestamp parsing and the working frame

The date and time are read in whatever form they arrive and joined into one timestamp, which becomes the index of the working table `df`.


In [4]:
def _parse_time_of_day(series):
    # Time of day may be a quoted string ('08:00:47'), a plain time string, a datetime,
    # or an Excel fraction of a day. Try each in turn.
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_timedelta(series, unit="D")
    s = series.astype(str).str.strip().str.strip("'\"")
    td = pd.to_timedelta(s, errors="coerce")
    if td.notna().any():
        return td
    dt = pd.to_datetime(s, errors="coerce")
    return dt - dt.dt.normalize()

# --- Build the working frame and its timestamp index --------------------------
df = raw.copy()

ts_col = resolve("timestamp")
if ts_col is not None:
    index = pd.to_datetime(df[ts_col], errors="coerce")
    source = f"combined column '{ts_col}'"
    df = df.drop(columns=[ts_col])
else:
    date_col = resolve("date", required=True)
    time_col = resolve("time")
    if pd.api.types.is_numeric_dtype(df[date_col]):
        # Plain numbers in a date column are Excel serial days.
        date_part = pd.to_datetime(df[date_col], unit="D", origin="1899-12-30", errors="coerce")
    else:
        date_part = pd.to_datetime(df[date_col], errors="coerce")
    date_part = date_part.dt.normalize()
    if time_col is not None and time_col != date_col:
        index = date_part + _parse_time_of_day(df[time_col])
        source = f"'{date_col}' plus '{time_col}'"
        df = df.drop(columns=[c for c in {date_col, time_col} if c in df.columns])
    else:
        index = date_part
        source = f"'{date_col}' alone (no separate time column found)"
        df = df.drop(columns=[date_col])

if index.isna().all():
    raise ValueError("could not parse any timestamps; check the date and time columns "
                     "or set COLUMN_OVERRIDES in the config cell")
df.index = pd.DatetimeIndex(index, name="timestamp")
df = df[~df.index.isna()].sort_index()

# A short confirmation that this step ran (useful now the section is split into steps).
print(f"Working frame built: {len(df)} rows, timestamp index from {source}.")


Working frame built: 7147 rows, timestamp index from 'Date' plus 'Time UTC(NC2)'.


#### Column typing: flags versus numbers

Columns that only ever hold zero and one (and are named like a mode) become true and false on/off flags; every other column is forced to a number so stray text becomes a missing value.


In [5]:
# --- Column types -------------------------------------------------------------
# A column is an on/off flag only if its name looks like one (mode, flag, status,
# enable) and its values are binary. The name test stops a setpoint that happens to be
# constant or binary in one capture from being mistaken for a flag.
def _is_flag(name, col):
    looks_like_flag = any(k in _norm(name) for k in ["mode", "flag", "status", "enable"])
    vals = set(pd.unique(col.dropna()))
    binary = 0 < len(vals) <= 2 and vals <= {0, 1, True, False, 0.0, 1.0, "0", "1"}
    return looks_like_flag and binary

mode_cols = [c for c in df.columns if _is_flag(c, df[c])]
df[mode_cols] = df[mode_cols].astype(float).astype(bool)

measure_cols = [c for c in df.columns if c not in mode_cols]
df[measure_cols] = df[measure_cols].apply(pd.to_numeric, errors="coerce")

# A short confirmation that this step ran (useful now the section is split into steps).
print(f"Column types set: {len(mode_cols)} on/off flag columns, {len(measure_cols)} numeric channels.")


Column types set: 6 on/off flag columns, 15 numeric channels.


#### Shared helpers for the test sections

Small functions every test reuses: finding the on periods of a flag, padding a window for the plot, and drawing the dotted event markers. Keeping them here means each test reads the same way.


In [6]:
# --- Shared helpers reused by every test section ------------------------------
def scope_for(test_name):
    # The part of the record a test should scan. By default this is the whole record. If
    # the config cell gave this test a (start, end) window in EVENT_WINDOWS, only that span
    # is returned. Having this here replaces a dense one-line slice in every test with a
    # single plainly named step.
    window = EVENT_WINDOWS.get(test_name)
    if window and all(window):
        start, end = window
        return df.loc[start:end]
    return df

def on_segments(flag):
    # Every period where an on/off flag is on, as a list of (start, end) times. start is
    # the first on sample; end is where it goes off again. This is how each test finds
    # all of its windows by itself, so a test repeated in one sheet gives one graph each.
    flag = flag.astype(bool)
    spans, start = [], None
    for ts, v in flag.items():
        if v and start is None:
            start = ts
        elif not v and start is not None:
            spans.append((start, ts)); start = None
    if start is not None:
        spans.append((start, flag.index[-1]))
    return spans

def window_around(start, end, before="90s", after="90s"):
    # A slice of df padded around a window and clipped to the data, so the setpoint-sent
    # lead-in and the recovery tail are both visible on the plot.
    lo = max(df.index[0], start - pd.Timedelta(before))
    hi = min(df.index[-1], end + pd.Timedelta(after))
    return df.loc[lo:hi]

def mark_events(ax, events, y, gap=6.0, fontsize=8.5):
    # Draw a dotted vertical line at each (time, colour, label) and write the label above
    # it, lifting every second one so neighbouring labels do not overlap.
    texts = []
    for i, (ts, colour, text) in enumerate(events):
        ax.axvline(ts, color=colour, ls=":", lw=1.3)
        texts.append(ax.annotate(text, xy=(ts, y), xytext=(ts, y + 2 + (i % 2) * gap),
                    ha="center", va="bottom", fontsize=fontsize, color=colour, fontweight="bold"))
    return texts

def mark_steps(ax, events, fontsize=8.2, lift=30):
    # Like mark_events but places the time label with point offsets instead of data-unit
    # offsets, so it reads correctly on a small-range panel (kilovolts, MVAr, power factor)
    # as well as on the megawatt panels. Labels are flat, and each one is lifted onto a stacked
    # level chosen so it never lands too close in time to another label already on that level.
    # Returns how many levels were used, so the caller can give the title a matching pad
    # (title_pad_for) and keep the lifted labels clear of it. Call only after the y limits are set.
    if not events:
        return 1
    top = ax.get_ylim()[1]
    times = [pd.Timestamp(ts) for ts, _colour, _text in events]
    span_seconds = (max(times) - min(times)).total_seconds() or 1.0
    # Two labels closer than this in time would touch, so they go on different levels.
    minimum_gap_seconds = 0.06 * span_seconds
    last_time_on_level = {}                         # level -> the most recent time placed there
    level_of = {}
    for i in sorted(range(len(events)), key=lambda j: times[j]):
        level = 0
        while (level in last_time_on_level
               and (times[i] - last_time_on_level[level]).total_seconds() < minimum_gap_seconds):
            level += 1
        last_time_on_level[level] = times[i]
        level_of[i] = level
    for i, (ts, colour, text) in enumerate(events):
        ax.axvline(ts, color=colour, ls=":", lw=1.3)
        ax.annotate(text, xy=(ts, top), xytext=(0, 6 + level_of[i] * lift),
                    textcoords="offset points", ha="center", va="bottom",
                    fontsize=fontsize, color=colour, fontweight="bold")
    return max(level_of.values()) + 1


def title_pad_for(levels, lift=30):
    # The padding a title needs so it sits above a mark_steps label stack of this many levels.
    return 6 + (levels - 1) * lift + 30


def constant_segments(series):
    # Each run where a held value stays constant, as (start, end, value). Used to read each
    # commanded setpoint level so the measurement can be judged against it level by level.
    s = series.dropna()
    segments = []
    if s.empty:
        return segments
    seg_start = s.index[0]
    seg_value = s.iloc[0]
    prev_ts = s.index[0]
    for ts, value in s.items():
        if value != seg_value:
            segments.append((seg_start, prev_ts, float(seg_value)))
            seg_start = ts
            seg_value = value
        prev_ts = ts
    segments.append((seg_start, prev_ts, float(seg_value)))
    return segments

# A short confirmation that this step ran (useful now the section is split into steps).
def format_signed_percent(value):
    # A whole-number percentage carrying an explicit + or - sign, but a plain "0%" when it
    # rounds to zero, so a tiny negative never shows as the ugly "-0%".
    rounded = round(value)
    if rounded == 0:
        return "0%"
    return f"{rounded:+d}%"

print("Shared helpers ready: scope_for, on_segments, window_around, mark_events, mark_steps, format_signed_percent.")


Shared helpers ready: scope_for, on_segments, window_around, mark_events, mark_steps, format_signed_percent.


#### Resolution report

A plain-language check of what was resolved and parsed, with no raw rows or identifiers shown. Confirm each role points at the right column before reading the tests.


In [7]:
# --- Report (no raw rows or identifiers are printed) --------------------------
step = df.index.to_series().diff().median()
print(f"Timestamp index built from {source}")
print(f"Parsed {len(df)} rows spanning {df.index.min()} to {df.index.max()} {TIME_ZONE_LABEL}")
print(f"Median sample step: {step}")
print("Resolved channels:")
for role in ["poc_p", "sp_p", "ap_mode", "pg_mode", "ramp_up", "ramp_down",
             "f_control", "grid_freq", "droop_f", "delta_mode", "delta_sp", "available",
             "v_mode", "v_meas", "v_sp", "droop_v", "q_mode", "q_meas", "q_sp",
             "pf_mode", "pf_meas", "pf_sp", "agc_mode", "hi_limit", "lo_limit",
             "sentout", "generated", "sp_feedback"]:
    print(f"  {role:10} -> {resolve(role)}")
print(f"On/off flag columns: {mode_cols}")
print(f"Missing values after numeric coercion: {int(df[measure_cols].isna().sum().sum())}")


Timestamp index built from 'Date' plus 'Time UTC(NC2)'
Parsed 7147 rows spanning 2026-05-27 08:00:47 to 2026-05-27 09:59:53 UTC
Median sample step: 0 days 00:00:01
Resolved channels:
  poc_p      -> POC: P (MW)
  sp_p       -> SP: P (MW)
  ap_mode    -> Mode: Active Power
  pg_mode    -> Mode: Power Gradient
  ramp_up    -> SP:Ramp up (MW/min)
  ramp_down  -> SP:Ramp down (MW/min)
  f_control  -> f used by f control (during test)
  grid_freq  -> POC: Freq (Hz)
  droop_f    -> SP:Droop F (%)
  delta_mode -> Mode: p-Delta
  delta_sp   -> SP: P-Delta (%)
  available  -> None
  v_mode     -> Mode:V
  v_meas     -> POC: Average Voltage (KV)
  v_sp       -> SP:Voltage (kV)
  droop_v    -> SP:Droop V (%)
  q_mode     -> Mode:Q
  q_meas     -> POC: Q (MVAr)
  q_sp       -> SP: Q (MVAr)
  pf_mode    -> Mode:PF
  pf_meas    -> POC: PF
  pf_sp      -> SP: PF
  agc_mode   -> None
  hi_limit   -> None
  lo_limit   -> None
  sentout    -> None
  generated  -> None
  sp_feedback -> None
On/off flag c

## Curtailment, the absolute production constraint test

Curtailment is the grid operator capping how much power the plant may export. The plant is given a ceiling in megawatts (MW); when curtailment mode is switched on it must pull its output down to sit at or below that ceiling, then return to normal when the mode is switched off. This is the absolute production constraint test in the acceptance procedure [1], checked in three steps: the ceiling setpoint is sent while the mode is still off and the output should only acknowledge it, then the mode is switched on and the output should fall to the ceiling and hold, then the mode is switched off and the output should recover.

The cell below finds every standalone curtailment window in the record, meaning curtailment mode on while the power gradient limiter is off, and draws one graph per window. For each it reads the ceiling from the data, marks the setpoint-sent, mode-on and mode-off moments with their times, and prints findings for the three checks.

#### Curtailment: find the windows

Resolve the channels this test needs and collect every standalone curtailment window (mode on while the power-gradient limiter is off).


In [8]:
# =============================================================================
# CURTAILMENT TEST  (Absolute Production Constraint, NCSS Section 7, p.10)
#
# What the plant must do, straight from the acceptance procedure:
#   CHECK 1  Ceiling setpoint sent while the mode is still OFF -> output should NOT
#            move yet (the command is received but inert).
#   CHECK 2  Curtailment mode ON  -> output should pull down to sit AT OR BELOW the
#            commanded ceiling.
#   CHECK 3  Curtailment mode OFF -> output should return to its earlier level.
#
# A "window" is one occurrence of the test. This cell finds every standalone occurrence
# (curtailment mode on while the power-gradient limiter is off); the next two cells
# assess and then draw one graph and one set of findings per window. Every value is
# read from the data.
# =============================================================================
import matplotlib.pyplot as plt    # draws the figures
import matplotlib.dates as mdates  # formats the time axis as HH:MM:SS

# The channels this test needs, found by role (never by a fixed column name).
power_col    = resolve("poc_p", required=True)   # measured active power
setpoint_col = resolve("sp_p", required=True)    # active power setpoint (the ceiling)
curtail_col  = resolve("ap_mode", required=True) # curtailment mode on/off flag
gradient_col = resolve("pg_mode")                # power-gradient flag (used to exclude its windows)

# Find every standalone curtailment window. A window counts for this test only if the
# power-gradient limiter is off during it; windows where the gradient limiter is mostly on
# belong to the power-gradient test instead, so they are skipped here.
scope = scope_for("curtailment")                 # the part of the record this test scans
curtailment_windows = []
for start, end in on_segments(scope[curtail_col]):
    if gradient_col is not None:
        gradient_flag = scope[gradient_col].astype(bool).loc[start:end]
        gradient_mostly_on = gradient_flag.mean() >= 0.5
    else:
        gradient_mostly_on = False               # this sheet has no gradient flag at all
    if not gradient_mostly_on:
        curtailment_windows.append((start, end))

print(f"Standalone curtailment windows found: {len(curtailment_windows)}")
if not curtailment_windows:
    print("Curtailment mode is never on without power gradient here, so there is nothing to plot.")


Standalone curtailment windows found: 2


#### Curtailment: assess each window

Decide, purely from the data, the three procedure moments and whether each acceptance check passes. No plotting, just the judging, with the verdicts printed.


In [9]:
# The acceptance tolerances for this test (how far over the ceiling still counts as
# curtailed, how far short of the earlier level still counts as returned, and how close in
# time the setpoint and mode count as one step) are read from the config cell, where they
# are explained. They are notebook reading aids, not grid-code values.

def assess_curtailment_window(mode_on, window_end):
    # Work out, purely from the measured data, the three procedure moments and whether each
    # acceptance check passes. No plotting happens here, only the judging.
    win      = window_around(mode_on, window_end)   # padded slice around this window
    curtail  = win[curtail_col].astype(bool)
    power    = win[power_col]                        # measured active power
    setpoint = win[setpoint_col]

    # The three procedure moments, located in the data.
    later_off = curtail.loc[mode_on:].index[~curtail.loc[mode_on:]]
    mode_off  = later_off[0] if len(later_off) else None     # when the mode goes off again
    sp_changes = setpoint.ne(setpoint.shift())
    sp_changes.iloc[0] = False                               # first sample is the window edge, not a change
    sent_steps = setpoint.index[sp_changes & (setpoint.index <= mode_on)]
    sp_sent   = sent_steps[-1] if len(sent_steps) else None  # the setpoint that set the ceiling
    ceiling   = setpoint.loc[mode_on]
    sent_with_mode = sp_sent is not None and abs(mode_on - sp_sent) <= pd.Timedelta(seconds=CURTAIL_SENT_TOGETHER_SECONDS)

    # CHECK 2: output held at or below the ceiling, within the ceiling tolerance.
    held_output = float(power[curtail].median())             # output while curtailment is on
    ceiling_tolerance = max(CURTAIL_CEILING_TOLERANCE_MW, CURTAIL_CEILING_TOLERANCE_FRACTION * abs(ceiling))
    check2_curtailed = held_output <= ceiling + ceiling_tolerance

    # CHECK 3: output returned to its earlier level after mode off, within the return tolerance.
    before_level = float(power[power.index < mode_on].median()) if (power.index < mode_on).any() else None
    after_series = power[power.index > mode_off] if mode_off is not None else power.iloc[0:0]
    after_level  = float(after_series.median()) if len(after_series) else None
    if before_level is None or after_level is None:
        check3_returned = False                              # recovery not captured in this window
    else:
        return_tolerance = max(CURTAIL_RETURN_TOLERANCE_MW, CURTAIL_RETURN_TOLERANCE_FRACTION * abs(before_level))
        check3_returned = after_level >= before_level - return_tolerance

    return {
        "mode_on": mode_on, "window_end": window_end, "ceiling": ceiling,
        "mode_off": mode_off, "sp_sent": sp_sent, "sent_with_mode": sent_with_mode,
        "held_output": held_output, "before_level": before_level, "after_level": after_level,
        "check2_curtailed": check2_curtailed, "check3_returned": check3_returned,
        "power": power, "setpoint": setpoint, "curtail": curtail,
    }

# Assess every window once; the next cell draws and narrates these results.
curtailment_results = [assess_curtailment_window(start, end) for start, end in curtailment_windows]
for n, result in enumerate(curtailment_results, start=1):
    check2 = "pass" if result["check2_curtailed"] else "fail"
    if result["mode_off"] is None or result["after_level"] is None:
        check3 = "not captured"
    else:
        check3 = "pass" if result["check3_returned"] else "fail"
    print(f"Window {n}: ceiling {result['ceiling']:.0f} MW at {result['mode_on']:%H:%M:%S}  ->  curtail check {check2}, return check {check3}")


Window 1: ceiling 35 MW at 08:15:31  ->  curtail check pass, return check pass
Window 2: ceiling 25 MW at 08:48:59  ->  curtail check pass, return check pass


#### Curtailment: draw and report each window

Draw one figure per window with every setpoint and moment labelled, and print the findings narrative for the three checks.


In [10]:
def draw_and_report_curtailment(result, n, total):
    # Draw the figure for one assessed window and print its findings narrative.
    power    = result["power"]
    setpoint = result["setpoint"]
    ceiling  = result["ceiling"]
    mode_on  = result["mode_on"]
    mode_off = result["mode_off"]
    sp_sent  = result["sp_sent"]
    sent_with_mode = result["sent_with_mode"]
    held_output    = result["held_output"]
    before_level   = result["before_level"]
    after_level    = result["after_level"]

    # --- Draw the graph ------------------------------------------------------
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(power.index, power, color="#1f77b4", lw=1.6, label="active power (measured)")
    ax.plot(setpoint.index, setpoint, color="#d62728", lw=1.8, ls="--", drawstyle="steps-post",
            label="active power setpoint (ceiling)")
    # value label on each setpoint level that is within view
    for ts, val in setpoint[setpoint.ne(setpoint.shift())].items():
        if power.min() - 5 <= val <= power.max() + 5:
            ax.annotate(f"{val:.0f} MW", xy=(ts, val), xytext=(3, 4), textcoords="offset points",
                        ha="left", va="bottom", fontsize=7.5, color="#d62728",
                        bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.6))

    # mark each moment with its time and what should happen there
    events = []
    if sent_with_mode:
        events.append((mode_on, "#ff7f0e", f"{mode_on:%H:%M:%S}\nsetpoint {ceiling:.0f} MW sent and mode ON together"))
    else:
        if sp_sent is not None:
            events.append((sp_sent, "#7f7f7f", f"{sp_sent:%H:%M:%S}\nsetpoint sent to {ceiling:.0f} MW\n(mode still off, output should hold)"))
        events.append((mode_on, "#ff7f0e", f"{mode_on:%H:%M:%S}\ncurtailment mode ON\n(output should fall to the ceiling)"))
    if mode_off is not None:
        events.append((mode_off, "#2ca02c", f"{mode_off:%H:%M:%S}\ncurtailment mode OFF\n(output should recover)"))
    mark_events(ax, events, power.max(), gap=6, fontsize=8.5)

    ax.set_ylim(power.min() - 4, power.max() + 22)
    ax.set_xlabel(f"Time ({TIME_ZONE_LABEL})")
    ax.set_ylabel("Active power (MW)")
    ax.set_title(f"{SITE_NAME} curtailment test {n} of {total}, ceiling {ceiling:.0f} MW at {mode_on:%H:%M:%S}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.legend(loc="upper right", framealpha=0.9)
    out_path = OUTPUT_DIR / f"{SITE_SLUG}_curtailment_{mode_on:%H%M%S}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    # --- Findings, written straight from the checks --------------------------
    story = []
    if sent_with_mode:                                                       # CHECK 1
        story.append(f"The {ceiling:.0f} MW setpoint and the mode were switched on together at "
                     f"{mode_on:%H:%M:%S}, so there is no separate sent-but-inert phase here.")
    elif sp_sent is not None:
        story.append(f"Check 1: the {ceiling:.0f} MW setpoint was sent at {sp_sent:%H:%M:%S} while the mode "
                     f"was still off and the output kept running, so the command was received but inert.")
    else:
        story.append("The setpoint was already at its ceiling before this window, so the moment it was sent "
                     "is not captured here.")
    if result["check2_curtailed"]:                                          # CHECK 2
        story.append(f"Check 2 pass: after mode ON the output settled near {held_output:.0f} MW, at or below "
                     f"the {ceiling:.0f} MW ceiling, so the plant was curtailed to the ceiling.")
    else:
        story.append(f"Check 2 fail: after mode ON the output stayed near {held_output:.0f} MW, above the "
                     f"{ceiling:.0f} MW ceiling, so it was not curtailed to the ceiling.")
    if mode_off is None:                                                    # CHECK 3
        story.append("Curtailment mode is still on at the end of this window, so the return to full output "
                     "is not captured here.")
    elif after_level is None:
        story.append(f"Mode OFF at {mode_off:%H:%M:%S} sits at the window edge, so recovery is not captured here.")
    elif result["check3_returned"]:
        story.append(f"Check 3 pass: after mode OFF at {mode_off:%H:%M:%S} the output recovered to about "
                     f"{after_level:.0f} MW, back near its earlier {before_level:.0f} MW, so it returned to full output.")
    else:
        story.append(f"Check 3 fail: after mode OFF at {mode_off:%H:%M:%S} the output recovered only to about "
                     f"{after_level:.0f} MW, below its earlier {before_level:.0f} MW, so it did not fully return.")

    print(f"\nCurtailment window {n} of {total}  ceiling {ceiling:.0f} MW  figure {out_path.name}")
    for k, line in enumerate(story, start=1):
        print(f"  {k}. {line}")
    plt.show()

# One graph and one set of findings per assessed window.
for n, result in enumerate(curtailment_results, start=1):
    draw_and_report_curtailment(result, n, len(curtailment_results))



Curtailment window 1 of 2  ceiling 35 MW  figure hartebeesthoek_curtailment_081531.png
  1. Check 1: the 35 MW setpoint was sent at 08:15:11 while the mode was still off and the output kept running, so the command was received but inert.
  2. Check 2 pass: after mode ON the output settled near 35 MW, at or below the 35 MW ceiling, so the plant was curtailed to the ceiling.
  3. Check 3 pass: after mode OFF at 08:16:11 the output recovered to about 48 MW, back near its earlier 50 MW, so it returned to full output.


C:\Users\erick\AppData\Local\Temp\ipykernel_22136\688421976.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Curtailment window 2 of 2  ceiling 25 MW  figure hartebeesthoek_curtailment_084859.png
  1. Check 1: the 25 MW setpoint was sent at 08:48:43 while the mode was still off and the output kept running, so the command was received but inert.
  2. Check 2 pass: after mode ON the output settled near 25 MW, at or below the 25 MW ceiling, so the plant was curtailed to the ceiling.
  3. Check 3 pass: after mode OFF at 08:49:45 the output recovered to about 45 MW, back near its earlier 49 MW, so it returned to full output.


## Power gradient constraint test

The power gradient constraint limits how fast the plant may change its output. Instead of jumping straight to a new setpoint, the plant must move towards it at a commanded rate, measured in megawatts per minute (MW/min), with a separate limit for ramping up and ramping down. The procedure [1] sets a down rate, switches power gradient mode on, commands a lower setpoint and checks the plant descends at that rate, then sets an up rate, commands a higher setpoint and checks the plant climbs at that rate.

The cell below finds every power gradient window in the record and draws one graph per window. While the mode is engaged it measures the slope the plant follows for each setpoint change by least-squares and compares it against the commanded limit for that direction. The measured ramp is drawn as a guide line, the curtailment mode that actually drives each move is marked with its on and off times along the bottom, and the findings give the observed rate beside the commanded one, anchored to the interval it was measured over.

#### Power gradient: find the windows

Resolve the channels and collect every period the power-gradient mode is on.


In [11]:
# =============================================================================
# POWER GRADIENT TEST  (Ramp Rate Constraint, NCSS Section 9, p.10)
#
# What the plant must do, straight from the acceptance procedure:
#   With power gradient mode ON, when the active power setpoint changes the plant must
#   move to the new setpoint NO FASTER than the commanded ramp rate (MW/min). There is
#   a separate limit for ramping up and for ramping down.
#   CHECK  for each setpoint change: measured ramp rate vs the commanded limit
#          (reported as close to / faster than / slower than the limit).
#
# A "window" is one occurrence (one power-gradient-on period); one graph each. The
# curtailment mode is what physically drives the plant to each setpoint, so its on and
# off times are marked along the bottom for context. The next three cells measure a
# ramp, assess each window, then draw it.
# =============================================================================
import numpy as np                # least-squares straight-line fit for the ramp slope
import matplotlib.pyplot as plt   # draws the figures
import matplotlib.dates as mdates # formats the time axis as HH:MM:SS

# The channels this test needs, found by role.
power_col      = resolve("poc_p", required=True)    # measured active power
setpoint_col   = resolve("sp_p", required=True)     # active power setpoint
gradient_col   = resolve("pg_mode", required=True)  # power gradient mode on/off flag
curtail_col    = resolve("ap_mode")                 # curtailment flag (drives the moves)
up_limit_col   = resolve("ramp_up")                 # commanded up ramp limit (MW/min)
down_limit_col = resolve("ramp_down")               # commanded down ramp limit (MW/min)

# Find every power gradient window (one period of the mode being on).
scope = scope_for("power_gradient")
power_gradient_windows = on_segments(scope[gradient_col])

print(f"Power gradient windows found: {len(power_gradient_windows)}")
if not power_gradient_windows:
    print("Power gradient mode is never on here, so there is nothing to plot.")


Power gradient windows found: 2


#### Power gradient: measure one ramp

The maths on its own: fit a straight line over the middle of a move (10 to 90 percent) and report its slope in MW per minute.


In [12]:
# The ramp-measurement constants (the 10 to 90 percent fit window and the smallest move
# that counts as a ramp) are read from the config cell, where they are explained.

def measure_ramp(power_segment, start_value, target_value, command_time=None):
    """Measure a single ramp: how fast the plant moved and how long the move took.

    The rate is the slope of a straight line fitted over the steady middle of the move (10
    percent risen to 90 percent risen), which skips the dead time before it starts and the
    flattening as it arrives, so the slope is the true ramp rate. The duration is measured
    separately: the time from when the plant could first move (command_time, the later of the
    setpoint being sent and curtailment being on, passed in by find_ramps) until the power
    first reaches the new target. Returns a small labelled result, or None if the move is too
    small to be a ramp.
    """
    total_move = target_value - start_value
    if abs(total_move) < RAMP_MIN_MOVE_MW:
        return None

    # The plant can only ramp once the setpoint is sent and curtailment is on, so the move is
    # measured only from command_time onward. This keeps both the fitted slope and the duration
    # from counting any drift before the plant was actually able to move.
    if command_time is None:
        command_time = power_segment.index[0]
    power_segment = power_segment.loc[command_time:]
    if len(power_segment) < 2:
        return None

    # --- Rate: slope over the 10 to 90 percent middle of the move ------------
    ten_percent_point = start_value + RAMP_RISE_LOW_FRACTION * total_move
    ninety_percent_point = start_value + RAMP_RISE_HIGH_FRACTION * total_move
    if total_move > 0:
        passed_ten = power_segment.index[power_segment >= ten_percent_point]
        passed_ninety = power_segment.index[power_segment >= ninety_percent_point]
    else:
        passed_ten = power_segment.index[power_segment <= ten_percent_point]
        passed_ninety = power_segment.index[power_segment <= ninety_percent_point]
    if len(passed_ten) > 0:
        ramp_start = passed_ten[0]
    else:
        ramp_start = power_segment.index[0]
    passed_ninety = passed_ninety[passed_ninety > ramp_start]
    if len(passed_ninety) > 0:
        ramp_end = passed_ninety[0]
    else:
        ramp_end = power_segment.index[-1]

    ramp = power_segment.loc[ramp_start:ramp_end]
    if len(ramp) < 2:
        return None
    minutes = (ramp.index - ramp.index[0]).total_seconds().to_numpy() / 60.0
    slope, intercept = np.polyfit(minutes, ramp.to_numpy(), 1)

    # --- Duration: from when the plant could first move to reaching the target -
    # command_time (set above) is when the plant was first able to ramp. The move ends when the
    # power first reaches the new target.
    if total_move > 0:
        reached_target = power_segment.index[power_segment >= target_value]
    else:
        reached_target = power_segment.index[power_segment <= target_value]
    if len(reached_target) > 0:
        arrival_time = reached_target[0]
    else:
        arrival_time = power_segment.index[-1]

    return {
        "rate": float(slope),                                # MW per minute, over the middle
        "t_start": ramp_start,                               # 10 percent point (for the guide line)
        "p_start": float(intercept + slope * minutes[0]),
        "t_end": ramp_end,                                   # 90 percent point (for the guide line)
        "p_end": float(intercept + slope * minutes[-1]),
        "command_time": command_time,                        # when the setpoint was commanded
        "arrival_time": arrival_time,                        # when the power first reached the target
        "duration": arrival_time - command_time,             # how long the move took, delay included
    }

def format_duration(span):
    # A short, plain reading of a length of time, for example "1 min 40 s" or "45 s".
    total_seconds = int(round(span.total_seconds()))
    minutes, seconds = divmod(total_seconds, 60)
    if minutes:
        return f"{minutes} min {seconds} s"
    return f"{seconds} s"

print("Ramp measurement helpers ready: measure_ramp (rate over 10 to 90 percent, plus move duration) and format_duration.")


Ramp measurement helpers ready: measure_ramp (rate over 10 to 90 percent, plus move duration) and format_duration.


#### Power gradient: assess each window

For each window, find every setpoint change and judge its measured ramp rate against the commanded up or down limit. No plotting, just the verdicts.


In [13]:
# The match tolerance (how close to the commanded limit still reads as "close to the
# limit") is read from the config cell. The report always states the objective figures
# (commanded limit, measured rate, and the percentage difference) so the reader can judge
# acceptability directly; the band only adds a plain convenience word (close / faster /
# slower) on top.

def find_ramps(win, gradient_on, active_until):
    """One entry per setpoint change while the mode is engaged, with its measured rate."""
    power = win[power_col]
    setpoint = win[setpoint_col]

    setpoint_changes = setpoint.ne(setpoint.shift())
    setpoint_changes.iloc[0] = False                         # first sample is the window edge
    step_times = setpoint.index[setpoint_changes]
    step_times = step_times[(step_times >= gradient_on) & (step_times <= active_until)]

    ramps = []
    for i, step_time in enumerate(step_times):
        target = setpoint.loc[step_time]
        start_value = power.loc[step_time]

        # Look at the power from this step until the next step (or the mode going off).
        if i + 1 < len(step_times):
            next_step = step_times[i + 1]
        else:
            next_step = active_until

        # The plant can only ramp once curtailment (the enabling mode) is on, so the move
        # begins at the later of the setpoint command and curtailment coming on. For the first
        # move curtailment usually comes on after the setpoint; for later moves it is already
        # on, so this reduces to the setpoint time.
        enable_time = step_time
        if curtail_col is not None:
            curtail_on = win[curtail_col].astype(bool).loc[step_time:next_step]
            on_times = curtail_on.index[curtail_on]
            if len(on_times) > 0:
                enable_time = max(step_time, on_times[0])
        measured = measure_ramp(power.loc[step_time:next_step], start_value, target, enable_time)
        if measured is None:
            continue

        # Which way did it move, and which commanded limit applies?
        if target < start_value:
            direction = "down"
            limit_col = down_limit_col
        else:
            direction = "up"
            limit_col = up_limit_col
        limit = win[limit_col].loc[step_time] if limit_col else None

        # When was that rate limit last set, at or before this setpoint was sent? The procedure
        # sets the rate first, then sends the setpoint. Read from the whole record so a limit set
        # before this window is still found.
        limit_sent = None
        if limit_col is not None:
            limit_series = df[limit_col]
            limit_changes = limit_series.ne(limit_series.shift())
            limit_changes.iloc[0] = True
            set_before = limit_series.index[limit_changes & (limit_series.index <= step_time)]
            limit_sent = set_before[-1] if len(set_before) else None

        ramp = dict(measured)                                # copy the measured result
        ramp.update(ts=step_time, target=target, direction=direction, limit=limit,
                    limit_sent=limit_sent)
        ramps.append(ramp)
    return ramps

def judge_ramp_against_limit(ramp):
    # Compare one ramp's measured rate to its commanded limit and return a plain verdict.
    if ramp["limit"] is None:
        return None
    gap = abs(ramp["rate"]) - ramp["limit"]
    if abs(gap) <= RAMP_MATCH_TOLERANCE_FRACTION * ramp["limit"]:
        return "close to the limit"
    elif gap > 0:
        return "faster than the limit"
    else:
        return "slower than the limit"

def assess_power_gradient_window(gradient_on, window_end):
    # Find the ramps in one window and judge each, with no plotting. Returns everything the
    # draw step needs.
    win      = window_around(gradient_on, window_end)
    gradient = win[gradient_col].astype(bool)
    power    = win[power_col]
    setpoint = win[setpoint_col]

    # When does the mode go off again?
    still_off = gradient.loc[gradient_on:].index[~gradient.loc[gradient_on:]]
    gradient_off = still_off[0] if len(still_off) > 0 else None
    active_until = gradient_off if gradient_off is not None else win.index[-1]

    ramps = find_ramps(win, gradient_on, active_until)
    for ramp in ramps:
        ramp["verdict"] = judge_ramp_against_limit(ramp)
        if ramp["limit"]:
            ramp["difference_percent"] = (abs(ramp["rate"]) - ramp["limit"]) / ramp["limit"] * 100.0
        else:
            ramp["difference_percent"] = None

    return {
        "gradient_on": gradient_on, "gradient_off": gradient_off, "active_until": active_until,
        "ramps": ramps, "power": power, "setpoint": setpoint, "win": win,
    }

# Assess every window once; the next cell draws and narrates these results.
power_gradient_results = [assess_power_gradient_window(start, end) for start, end in power_gradient_windows]
for n, result in enumerate(power_gradient_results, start=1):
    print(f"Window {n}: power gradient on at {result['gradient_on']:%H:%M:%S}, {len(result['ramps'])} ramp(s)")
    for ramp in result["ramps"]:
        took_txt = format_duration(ramp["duration"])
        if ramp["limit"] is None:
            print(f"   {ramp['direction']} ramp to {ramp['target']:.0f} MW: limit not recorded, "
                  f"measured {abs(ramp['rate']):.1f} MW/min, took {took_txt}")
        else:
            print(f"   {ramp['direction']} ramp to {ramp['target']:.0f} MW: limit {ramp['limit']:.0f} MW/min, "
                  f"measured {abs(ramp['rate']):.1f} MW/min, difference {format_signed_percent(ramp['difference_percent'])} "
                  f"({ramp['verdict']}), took {took_txt}")


Window 1: power gradient on at 08:19:11, 2 ramp(s)
   down ramp to 26 MW: limit 20 MW/min, measured 20.0 MW/min, difference 0% (close to the limit), took 1 min 13 s
   up ramp to 36 MW: limit 10 MW/min, measured 10.2 MW/min, difference +2% (close to the limit), took 46 s
Window 2: power gradient on at 08:51:41, 2 ramp(s)
   down ramp to 39 MW: limit 10 MW/min, measured 7.8 MW/min, difference -22% (slower than the limit), took 47 s
   up ramp to 44 MW: limit 5 MW/min, measured 5.5 MW/min, difference +11% (close to the limit), took 1 min 14 s


#### Power gradient: draw and report each window

Draw one figure per window with the measured slope, setpoints and mode times, and print the per-ramp findings.


In [14]:
def draw_and_report_power_gradient(result, n, total):
    # Draw the figure for one assessed window and print its findings narrative.
    gradient_on  = result["gradient_on"]
    gradient_off = result["gradient_off"]
    active_until = result["active_until"]
    ramps        = result["ramps"]
    power        = result["power"]
    setpoint     = result["setpoint"]
    win          = result["win"]

    # --- Draw the graph ------------------------------------------------------
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(power.index, power, color="#5b9bd5", lw=2.6, label="active power (measured)")
    ax.plot(setpoint.index, setpoint, color="#d62728", lw=1.8, ls="--", drawstyle="steps-post",
            label="active power setpoint")
    ax.axvspan(gradient_on, active_until, color="#9467bd", alpha=0.10, label="power gradient mode ON")

    # the measured slope of each ramp, drawn as a thin guide line so the thicker blue measured
    # power shows through and you can see the fit sitting on the data
    for ramp in ramps:
        ax.plot([ramp["t_start"], ramp["t_end"]], [ramp["p_start"], ramp["p_end"]],
                color="#2ca02c", lw=1.3, alpha=0.95)

    # value label on each setpoint level
    for ts, val in setpoint[setpoint.ne(setpoint.shift())].items():
        ax.annotate(f"{val:.0f} MW", xy=(ts, val), xytext=(3, 4), textcoords="offset points",
                    ha="left", va="bottom", fontsize=7.5, color="#d62728",
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.6))

    # top markers: mode on/off and each ramp. The marker is just the command: when the setpoint
    # was sent and to what. The measured results (duration, rate vs limit, and when the rate limit
    # was set) are shown beside each ramp's line of best fit below, in one green block matching the fit line.
    top_events = [(gradient_on, "#7f7f7f", f"{gradient_on:%H:%M:%S}\npower gradient ON")]
    measured_texts = []  # the blocks beside each fit line; kept glued to the ramp
    for ramp in ramps:
        if ramp["limit"] is not None:
            rate_vs_limit = f"measured {abs(ramp['rate']):.1f} vs limit {ramp['limit']:.0f} MW/min ({format_signed_percent(ramp['difference_percent'])})"
        else:
            rate_vs_limit = f"measured {abs(ramp['rate']):.1f} MW/min (no limit recorded)"
        top_events.append((ramp["ts"], "#e06666",
                           f"{ramp['ts']:%H:%M:%S}\n{ramp['direction']} ramp to {ramp['target']:.0f} MW sent"))

        # the measured block beside the fit line: green to match the fit line, left-aligned so every line
        # starts directly under "took". Anchored at the start of the ramp and mirrored by
        # direction so it never crowds the sloped line or the target setpoint box: above the high
        # end for a down ramp, below the low end for an up ramp.
        measured_lines = [f"took {format_duration(ramp['duration'])}", rate_vs_limit]
        if ramp["limit_sent"] is not None:
            measured_lines.append(f"rate limit set at {ramp['limit_sent']:%H:%M:%S}")
        measured_text = "\n".join(measured_lines)
        if ramp["direction"] == "down":
            label_height, v_align, label_offset = max(ramp["p_start"], ramp["p_end"]), "bottom", 10
        else:
            label_height, v_align, label_offset = min(ramp["p_start"], ramp["p_end"]), "top", -10
        measured_texts.append(ax.annotate(measured_text, xy=(ramp["t_start"], label_height),
                    xytext=(4, label_offset), textcoords="offset points", ha="left", va=v_align,
                    fontsize=7.0, color="#2ca02c", fontweight="bold"))
    if gradient_off is not None:
        top_events.append((gradient_off, "#7f7f7f", f"{gradient_off:%H:%M:%S}\npower gradient OFF"))
    top_marker_texts = mark_events(ax, top_events, power.max(), gap=8, fontsize=8.2)

    # curtailment markers: ON is lifted into the empty headroom at the top (cleaner than below
    # the data); OFF stays just below the data. Both keep a full-height dotted line to their time.
    curtail_top_texts = []      # the ON labels, raised into the top headroom
    curtail_bottom_texts = []   # the OFF labels, below the data
    if curtail_col:
        for mode_start, mode_end in on_segments(win[curtail_col]):
            for ts, text in ((mode_start, "curtailment mode ON"), (mode_end, "curtailment mode OFF")):
                ax.axvline(ts, color="#ff7f0e", ls=":", lw=1.3)
                if "ON" in text:
                    curtail_top_texts.append(ax.annotate(f"{ts:%H:%M:%S}\n{text}", xy=(ts, power.max()),
                                xytext=(ts, power.max() + 24), ha="center", va="bottom", fontsize=7.8,
                                color="#ff7f0e", fontweight="bold"))
                else:
                    curtail_bottom_texts.append(ax.annotate(f"{ts:%H:%M:%S}\n{text}", xy=(ts, power.min()),
                                xytext=(ts, power.min() - 3), ha="center", va="top", fontsize=7.8,
                                color="#ff7f0e", fontweight="bold"))

    ax.set_ylim(power.min() - 14, power.max() + 34)
    ax.set_xlabel(f"Time ({TIME_ZONE_LABEL})")
    ax.set_ylabel("Active power (MW)")
    ax.set_title(f"{SITE_NAME} power gradient test {n} of {total}, ramping at the commanded rate at {gradient_on:%H:%M:%S}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.legend(loc="upper right", framealpha=0.9)

    # --- Keep labels from overlapping (generic, measured from the drawn text) -
    # The measured blocks stay glued to their ramps. Any movable curtailment label that overlaps
    # a block is nudged further down, and the y axis is stretched to keep it in view. Overlap is
    # read from the actual drawn text boxes, so this adapts to any capture rather than relying on
    # a fixed offset.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    nudge = max(1.0, 0.03 * (power.max() - power.min()))      # how far to move per try, in MW

    def overlaps_any(label, obstacles):
        label_box = label.get_window_extent(renderer)
        return any(label_box.overlaps(other.get_window_extent(renderer)) for other in obstacles)

    # ON labels rise until they clear the took blocks, the mode/ramp markers and the legend,
    # stretching the top of the axis as needed
    legend = ax.get_legend()
    top_obstacles = measured_texts + top_marker_texts + ([legend] if legend is not None else [])
    for label in curtail_top_texts:
        guard = 0
        while overlaps_any(label, top_obstacles) and guard < 80:
            x, y = label.get_position()
            label.set_position((x, y + nudge))
            low, high = ax.get_ylim()
            if y + 2 * nudge > high:
                ax.set_ylim(low, y + 2 * nudge)
            fig.canvas.draw()
            guard += 1

    # OFF labels sink until they clear the took blocks, stretching the bottom as needed
    for label in curtail_bottom_texts:
        guard = 0
        while overlaps_any(label, measured_texts) and guard < 80:
            x, y = label.get_position()
            label.set_position((x, y - nudge))
            low, high = ax.get_ylim()
            if y - 2 * nudge < low:
                ax.set_ylim(y - 2 * nudge, high)
            fig.canvas.draw()
            guard += 1

    out_path = OUTPUT_DIR / f"{SITE_SLUG}_power_gradient_{gradient_on:%H%M%S}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    # --- Findings: one line per ramp, with the time taken and the rate vs limit
    story = [f"Power gradient mode was switched on at {gradient_on:%H:%M:%S}, so the plant should move "
             "between setpoints at the commanded rate rather than as fast as it can."]
    for ramp in ramps:
        took_txt = format_duration(ramp["duration"])
        if ramp["limit"] is None:
            story.append(f"At {ramp['ts']:%H:%M:%S} the setpoint moved {ramp['direction']} to {ramp['target']:.0f} MW; "
                         f"it took about {took_txt} to get there (timed from when the plant could first move, once curtailment was on). "
                         f"The measured rate over the middle of the move was {abs(ramp['rate']):.1f} MW/min; no rate limit was recorded.")
        else:
            story.append(f"At {ramp['ts']:%H:%M:%S} the setpoint moved {ramp['direction']} to {ramp['target']:.0f} MW; "
                         f"it took about {took_txt} to get there (timed from when the plant could first move, once curtailment was on). "
                         f"The commanded limit was {ramp['limit']:.0f} MW/min and the measured rate over the middle of "
                         f"the move was {abs(ramp['rate']):.1f} MW/min, a difference of {format_signed_percent(ramp['difference_percent'])} "
                         f"({ramp['verdict']}, by the notebook's {RAMP_MATCH_TOLERANCE_FRACTION * 100:.0f} percent convenience band).")
    if not ramps:
        story.append("No setpoint change happens while power gradient is on, so the rate limit is not "
                     "exercised in this window.")
    if gradient_off is not None:
        story.append(f"Power gradient mode was released at {gradient_off:%H:%M:%S}.")
    else:
        story.append("Power gradient mode is still on at the end of this window.")

    print(f"\nPower gradient window {n} of {total}  figure {out_path.name}")
    for k, line in enumerate(story, start=1):
        print(f"  {k}. {line}")
    plt.show()

# One graph and one set of findings per assessed window.
for n, result in enumerate(power_gradient_results, start=1):
    draw_and_report_power_gradient(result, n, len(power_gradient_results))



Power gradient window 1 of 2  figure hartebeesthoek_power_gradient_081911.png
  1. Power gradient mode was switched on at 08:19:11, so the plant should move between setpoints at the commanded rate rather than as fast as it can.
  2. At 08:19:35 the setpoint moved down to 26 MW; it took about 1 min 13 s to get there (timed from when the plant could first move, once curtailment was on). The commanded limit was 20 MW/min and the measured rate over the middle of the move was 20.0 MW/min, a difference of 0% (close to the limit, by the notebook's 20 percent convenience band).
  3. At 08:22:06 the setpoint moved up to 36 MW; it took about 46 s to get there (timed from when the plant could first move, once curtailment was on). The commanded limit was 10 MW/min and the measured rate over the middle of the move was 10.2 MW/min, a difference of +2% (close to the limit, by the notebook's 20 percent convenience band).
  4. Power gradient mode was released at 08:23:27.


C:\Users\erick\AppData\Local\Temp\ipykernel_22136\2132730827.py:155: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Power gradient window 2 of 2  figure hartebeesthoek_power_gradient_085141.png
  1. Power gradient mode was switched on at 08:51:41, so the plant should move between setpoints at the commanded rate rather than as fast as it can.
  2. At 08:52:15 the setpoint moved down to 39 MW; it took about 47 s to get there (timed from when the plant could first move, once curtailment was on). The commanded limit was 10 MW/min and the measured rate over the middle of the move was 7.8 MW/min, a difference of -22% (slower than the limit, by the notebook's 20 percent convenience band).
  3. At 08:54:47 the setpoint moved up to 44 MW; it took about 1 min 14 s to get there (timed from when the plant could first move, once curtailment was on). The commanded limit was 5 MW/min and the measured rate over the middle of the move was 5.5 MW/min, a difference of +11% (close to the limit, by the notebook's 20 percent convenience band).
  4. Power gradient mode was released at 08:57:24.


C:\Users\erick\AppData\Local\Temp\ipykernel_22136\2132730827.py:155: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Frequency response

The grid behaves as one synchronised machine, and its **frequency** (in **hertz, Hz**) is how fast it turns; the South African grid runs at a nominal 50 Hz. Frequency rises when generation exceeds demand, so holding it near 50 Hz means constantly balancing the two across the **National Integrated Power System (NIPS)**.

When the frequency climbs too high there is too much generation, and a **renewable power plant (RPP)** like this one must reduce its output above a set point: the **over-frequency** response. Two settings shape it. The **deadband** is a band around 50 Hz inside which the plant does nothing, ignoring normal wobble. The **droop** sets how sharply the plant cuts its power as the frequency climbs: a smaller droop means a bigger cut for the same rise in frequency (the grid code defines it precisely as the percentage change in frequency that moves the plant across its full output range [2], SAGC Section 4, p.8, and lets it be set from 0 to 10 percent by the System Operator, SAGC 6.2(5) to 6.2(6), p.20). Everything is judged at the **point of connection (POC)** where the plant joins the grid, and the test is run by the **System Operator (SO)**.

This section checks the same test two ways: against the acceptance procedure in the test record [1] (response switched on, plant seen reacting outside the deadband, switched off), and against the South African Grid Code [2], which sets the exact reduction curve above 50.5 Hz (SAGC 6.1(2) and Figure 6, p.19) and the trip required above 51.5 Hz held for 4 seconds (SAGC 6.1(3), p.19). One note: this logger has no frequency-mode on or off flag, so the test is found directly from the swept control frequency that was injected to drive it.


### Checked against the test record procedure

The acceptance procedure [1] for frequency response is short: switch the response on, confirm the control was sent and the mode changed to on, observe the plant reacting to frequency deviations outside the deadband, then switch the response off. This logger carries no frequency-mode flag, so rather than a mode switching on and off, the test is read from the controlling frequency that was swept upward to provoke the plant. The cells below read the basic facts first, then find each over-frequency window in the record, draw one time plot per window, and report the findings.

In [15]:
# =============================================================================
# FREQUENCY RESPONSE  -  reading the test off the data (basic facts)
#
# This sheet has no frequency mode on/off flag, so before anything else we read the
# plain facts that tell us how the over-frequency test was driven and measured:
#   - the controlling frequency that was swept upward to provoke a response
#   - the recorded droop setting (read from the sheet, never assumed)
#   - the measured grid frequency at the point of connection
#   - a check that no frequency mode flag exists in this record
# =============================================================================

# 1. The channels this test needs, found by role (never by a fixed column name).
control_freq_col = resolve("f_control", required=True)  # injected control frequency (the stimulus)
grid_freq_col    = resolve("grid_freq", required=True)   # measured grid frequency at the POC
power_col        = resolve("poc_p", required=True)       # measured active power at the POC
droop_col        = resolve("droop_f")                    # recorded frequency droop setting (percent)

# 2. The swept controlling frequency: the signal that was driven up to provoke a response.
control_freq = df[control_freq_col].dropna()
print(f"Controlling frequency swept from {control_freq.min():.2f} Hz to {control_freq.max():.2f} Hz")

# 3. The recorded droop, read straight from the sheet (a single setting in this record).
#    Droop is defined in the SAGC Definitions (Section 4, p.8) and is set between 0 and 10
#    percent by the System Operator (SAGC 6.2(5), p.20); the value here is read, never assumed.
if droop_col is not None:
    droop_levels = df[droop_col].dropna().unique()
    if len(droop_levels) == 1:
        droop_percent = float(droop_levels[0])
        print(f"Recorded frequency droop: {droop_percent:.0f} percent (read from the sheet)")
    else:
        droop_percent = float(df[droop_col].dropna().median())
        print(f"Recorded frequency droop varies; using the median {droop_percent:.1f} percent")
else:
    droop_percent = None
    print("No droop column found in this record")

# 4. The measured grid frequency, to compare against the swept control signal later.
grid_freq = df[grid_freq_col].dropna()
print(f"Measured grid frequency ranged {grid_freq.min():.2f} Hz to {grid_freq.max():.2f} Hz")
print(f"Over-frequency point used for the checks: {F4_OVER_FREQUENCY_HZ:.1f} Hz (SAGC 6.1(2) and Figure 6, p.19)")

# 5. Is there a frequency mode on/off flag? List any mode flag whose name mentions frequency.
frequency_flags = [flag for flag in mode_cols if "freq" in _norm(flag)]
if frequency_flags:
    print(f"Frequency mode flag(s) present: {frequency_flags}")
else:
    print("No frequency mode flag exists in this record, so the test is found from the "
          "swept controlling frequency itself.")


Controlling frequency swept from 49.84 Hz to 51.90 Hz
Recorded frequency droop: 4 percent (read from the sheet)
Measured grid frequency ranged 49.82 Hz to 50.21 Hz
Over-frequency point used for the checks: 50.5 Hz (SAGC 6.1(2) and Figure 6, p.19)
No frequency mode flag exists in this record, so the test is found from the swept controlling frequency itself.


In [16]:
# =============================================================================
# FREQUENCY RESPONSE  -  finding the over-frequency windows
#
# A "window" is one occurrence of the test: one stretch where the controlling
# frequency sat at or above the over-frequency point, meaning the stimulus was
# applied. We keep only the clean windows: those with NO active-power constraint mode
# on (curtailment, power gradient or p-Delta), so that any power reduction we later
# see is attributable to frequency alone and not to another test running at the same
# time.
#
# For each clean window we measure, straight from the data:
#   - when the frequency crossed up through the over-frequency point, and back down
#   - the peak frequency reached, and when
#   - the reference power: the output just before the crossing (the reduction is
#     measured against this)
#   - the floor power: the lowest output held during the window
#   - when the output recovered back towards its reference
# =============================================================================

# 1. The modes that constrain active power; a window overlapping any of these is not
#    a clean frequency test and is set aside.
active_power_flag   = resolve("ap_mode")     # curtailment (absolute production constraint)
power_gradient_flag = resolve("pg_mode")     # power gradient (ramp rate limit)
delta_flags = [flag for flag in mode_cols if "delta" in _norm(flag)]   # p-Delta constraint
constraint_flags = [f for f in [active_power_flag, power_gradient_flag] if f] + delta_flags

# 2. Restrict to the configured scope (whole record unless limited in the config cell).
scope = scope_for("frequency")

# 3. Every stretch where the controlling frequency is at or above the over-frequency point.
above_threshold = scope[control_freq_col] >= F4_OVER_FREQUENCY_HZ
raw_crossings = on_segments(above_threshold)


def measure_reference_power(power, cross_up):
    """The output just before the frequency crossed the threshold.

    Uses the median of the half minute before the crossing so a single noisy sample
    cannot set the reference. Falls back to the value at the crossing if nothing
    earlier is captured.
    """
    before = power.loc[cross_up - pd.Timedelta(seconds=FREQ_REFERENCE_LOOKBACK_SECONDS):cross_up]
    before = before[before.index < cross_up]
    if len(before):
        return float(before.median())
    return float(power.loc[cross_up])


# FREQ_RECOVERY_FRACTION (config) is how close to its earlier level the output must climb to
# count as recovered. A notebook reading aid, not a grid-code figure: the test record only
# asks to see the output return after the frequency drops back below the point.

def find_recovery(power, cross_down, reference_power):
    """When the output first climbed back to within reach of its reference after the
    window, or None if it did not recover before the data ran out.
    """
    after = power.loc[cross_down:]
    after = after[after.index > cross_down]
    recovered = after.index[after >= FREQ_RECOVERY_FRACTION * reference_power]
    return recovered[0] if len(recovered) else None


# 4. Keep the clean windows and measure each one.
frequency_windows = []
for cross_up, cross_down in raw_crossings:
    overlaps_constraint = False
    for flag in constraint_flags:
        if scope[flag].astype(bool).loc[cross_up:cross_down].mean() >= 0.5:
            overlaps_constraint = True
    if overlaps_constraint:
        continue

    control_segment = scope[control_freq_col].loc[cross_up:cross_down]
    power_segment = df[power_col].loc[cross_up:cross_down]
    reference_power = measure_reference_power(df[power_col], cross_up)
    floor_power = float(power_segment.min())

    frequency_windows.append({
        "cross_up": cross_up,
        "cross_down": cross_down,
        "peak_freq": float(control_segment.max()),
        "peak_time": control_segment.idxmax(),
        "reference_power": reference_power,
        "floor_power": floor_power,
        "recovery_time": find_recovery(df[power_col], cross_down, reference_power),
    })

# 5. Report what was found, honestly.
print(f"Stretches above {F4_OVER_FREQUENCY_HZ:.1f} Hz: {len(raw_crossings)}; "
      f"clean over-frequency windows (no constraint mode on): {len(frequency_windows)}")
for n, window in enumerate(frequency_windows, start=1):
    floor_percent = 100.0 * window["floor_power"] / window["reference_power"] if window["reference_power"] else float("nan")
    print(f"  Window {n}: {window['cross_up']:%H:%M:%S} to {window['cross_down']:%H:%M:%S}, "
          f"peak {window['peak_freq']:.2f} Hz, reference {window['reference_power']:.0f} MW, "
          f"floor {window['floor_power']:.0f} MW ({floor_percent:.0f} percent of reference)")
if not frequency_windows:
    print("No clean over-frequency window is present, so there is nothing to plot.")


Stretches above 50.5 Hz: 2; clean over-frequency windows (no constraint mode on): 2
  Window 1: 08:32:52 to 08:37:26, peak 51.90 Hz, reference 52 MW, floor 14 MW (26 percent of reference)
  Window 2: 09:04:35 to 09:07:37, peak 51.90 Hz, reference 48 MW, floor 13 MW (28 percent of reference)


In [17]:
# =============================================================================
# FREQUENCY RESPONSE  -  one time plot per over-frequency window
#
# For each window found above we draw two stacked panels that share a time axis:
#   TOP    the controlling frequency, with the over-frequency point marked as a
#          horizontal line; each key moment is a dotted line carrying its time.
#   BOTTOM the measured active power, with the reference level it should be judged
#          against, and the same key moments marked with their times and what should
#          happen there.
# The reader should be able to see the power fall as the frequency rises past the
# threshold and recover as it drops back, all on the graph itself.
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_frequency_window(window, n, total):
    win = window_around(window["cross_up"], window["cross_down"])   # padded slice for context
    control = win[control_freq_col]
    power = win[power_col]
    reference_power = window["reference_power"]

    # --- The key moments to mark on both panels ------------------------------
    # Recovery is only shown separately when it is clearly later than the drop back
    # below the threshold; when the two nearly coincide they are the same moment and
    # one marker keeps the graph readable.
    moments = [
        (window["cross_up"], "#ff7f0e", "frequency crossed\n"
         f"{F4_OVER_FREQUENCY_HZ:.1f} Hz\n(reduction should begin)"),
        (window["peak_time"], "#d62728", f"peak {window['peak_freq']:.1f} Hz"),
        (window["cross_down"], "#2ca02c", f"back below\n{F4_OVER_FREQUENCY_HZ:.1f} Hz\n(output should recover)"),
    ]
    recovery = window["recovery_time"]
    if recovery is not None and (recovery - window["cross_down"]) > pd.Timedelta(seconds=FREQ_RECOVERY_MARK_GAP_SECONDS):
        moments.append((recovery, "#1f77b4", "output recovered"))

    # --- Draw the two panels -------------------------------------------------
    fig, (ax_freq, ax_power) = plt.subplots(2, 1, sharex=True, figsize=(12, 9),
                                            gridspec_kw={"height_ratios": [1, 1.15]})

    # Top panel: controlling frequency and the over-frequency point.
    ax_freq.plot(control.index, control, color="#9467bd", lw=1.8, label="controlling frequency")
    ax_freq.axhline(F4_OVER_FREQUENCY_HZ, color="#d62728", ls="--", lw=1.5,
                    label=f"over-frequency point {F4_OVER_FREQUENCY_HZ:.1f} Hz")
    # each moment is a dotted line with its time written flat above the line; every
    # second time is lifted a little further so neighbouring times never merge.
    for i, (ts, colour, _text) in enumerate(moments):
        ax_freq.axvline(ts, color=colour, ls=":", lw=1.3)
        lift_points = 5 + (i % 2) * 16
        ax_freq.annotate(f"{ts:%H:%M:%S}", xy=(ts, control.max()), xytext=(0, lift_points),
                         textcoords="offset points", ha="center", va="bottom",
                         fontsize=8.5, color=colour, fontweight="bold")
    ax_freq.set_ylim(control.min() - 0.15, control.max() + 1.6)
    ax_freq.set_ylabel("Frequency (Hz)")
    ax_freq.set_title(f"{SITE_NAME} frequency response window {n} of {total}, "
                      f"peak {window['peak_freq']:.1f} Hz at {window['cross_up']:%H:%M:%S}")
    ax_freq.legend(loc="upper right", framealpha=0.9)

    # Bottom panel: measured active power and its reference level.
    ax_power.plot(power.index, power, color="#1f77b4", lw=1.6, label="active power (measured)")
    ax_power.axhline(reference_power, color="#7f7f7f", ls="--", lw=1.3,
                     label=f"reference before crossing ({reference_power:.0f} MW)")
    floor_percent = 100.0 * window["floor_power"] / reference_power if reference_power else float("nan")
    ax_power.annotate(f"floor {window['floor_power']:.0f} MW ({floor_percent:.0f}% of reference)",
                      xy=(window["peak_time"], window["floor_power"]), xytext=(6, -4),
                      textcoords="offset points", ha="left", va="top", fontsize=8.5,
                      color="#1f77b4", fontweight="bold")
    mark_events(ax_power, moments, power.max(), gap=9, fontsize=8.2)
    # generous headroom so the legend and the lifted labels sit clear of the traces
    power_span = power.max() - power.min()
    ax_power.set_ylim(power.min() - 5, power.max() + max(38, power_span))
    ax_power.set_ylabel("Active power (MW)")
    ax_power.set_xlabel(f"Time ({TIME_ZONE_LABEL})")
    ax_power.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax_power.legend(loc="upper right", framealpha=0.9)

    out_path = OUTPUT_DIR / f"{SITE_SLUG}_frequency_response_{window['cross_up']:%H%M%S}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Frequency response window {n} of {total}  figure {out_path.name}")
    plt.show()


# One graph per window.
if not frequency_windows:
    print("No clean over-frequency window is present, so there is nothing to plot.")
for n, window in enumerate(frequency_windows, start=1):
    plot_frequency_window(window, n, len(frequency_windows))


Frequency response window 1 of 2  figure hartebeesthoek_frequency_response_083252.png


C:\Users\erick\AppData\Local\Temp\ipykernel_22136\3436146970.py:80: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Frequency response window 2 of 2  figure hartebeesthoek_frequency_response_090435.png


### What the time plots show

Both windows tell the same story. While the controlling frequency sits at 50 Hz the plant runs near its reference output, about 52 MW in the first window and 48 MW in the second. The instant the frequency is stepped up through the 50.5 Hz over-frequency point the active power drops hard, and within seconds it reaches a floor of about 14 MW, close to a quarter of where it started. It holds there for as long as the frequency stays elevated, through the climb to the 51.9 Hz peak and back down, then it returns to its reference the moment the frequency drops below 50.5 Hz.

So the test record checks are clearly met in both windows: the stimulus was applied and the plant responded, in the right direction, every time. Two things are worth carrying into the grid-code comparison that follows. First, the reduction is a single sharp step down to a floor rather than a gentle slope, and that shape is what the required droop curve will be weighed against. Second, the frequency that drove all of this is the injected control frequency, not the measured grid frequency, which never left its normal band near 50 Hz, so the response was exercised by the test signal exactly as a commissioning test should.

In [18]:
# =============================================================================
# FREQUENCY RESPONSE  -  findings against the test record procedure
#
# The acceptance procedure asks two plain things of this test [1]:
#   CHECK 1  Was the stimulus applied? The controlling frequency must be driven above
#            the over-frequency point so the plant has something to react to.
#   CHECK 2  Did the plant respond? Active power must fall while the frequency is above
#            the point, and return when it drops back below.
# These are the test-record checks only. Whether the SIZE of the reduction matches the
# grid code curve is judged in the next subsection.
# =============================================================================

# A clear response means the output fell by a margin the notebook treats as unambiguous (the
# FREQ_RESPONSE_MIN_DROP_* reading aids in the config cell). The test record only asks to
# observe the plant reacting outside the deadband, not by how much; the size of the reduction
# is judged against the grid-code curve in the next subsection.

if not frequency_windows:
    print("No over-frequency window was captured, so the frequency response test is not "
          "exercised in this record.")

for n, window in enumerate(frequency_windows, start=1):
    reference = window["reference_power"]
    floor = window["floor_power"]
    floor_percent = 100.0 * floor / reference if reference else float("nan")
    reduction = reference - floor

    # CHECK 1: was the stimulus applied?
    stimulus_applied = window["peak_freq"] >= F4_OVER_FREQUENCY_HZ
    # CHECK 2: did the plant respond? (a clear drop in output while the frequency was high)
    plant_responded = reduction > max(FREQ_RESPONSE_MIN_DROP_MW, FREQ_RESPONSE_MIN_DROP_FRACTION * reference)
    output_recovered = window["recovery_time"] is not None

    print(f"\nWindow {n}  {window['cross_up']:%H:%M:%S} to {window['cross_down']:%H:%M:%S}")
    if stimulus_applied:
        print(f"  1. Stimulus applied: the controlling frequency was driven to {window['peak_freq']:.1f} Hz, "
              f"above the {F4_OVER_FREQUENCY_HZ:.1f} Hz over-frequency point.")
    else:
        print(f"  1. Stimulus not applied: the controlling frequency only reached {window['peak_freq']:.1f} Hz, "
              f"not above the {F4_OVER_FREQUENCY_HZ:.1f} Hz point.")
    if plant_responded:
        print(f"  2. Plant responded: active power fell from about {reference:.0f} MW to a floor of "
              f"{floor:.0f} MW ({floor_percent:.0f} percent of reference) while the frequency was high.")
    else:
        print(f"  2. No clear response: active power stayed near {reference:.0f} MW while the frequency was high.")
    if output_recovered:
        print(f"  3. Output recovered to its reference at {window['recovery_time']:%H:%M:%S}, after the "
              "frequency dropped back below the point.")
    else:
        print("  3. The recovery is not captured before the data ends in this window.")

# Honest note about what actually drove the test.
if frequency_windows:
    print(f"\nMeasured grid frequency over the whole record stayed between {grid_freq.min():.2f} Hz and "
          f"{grid_freq.max():.2f} Hz, never above the {F4_OVER_FREQUENCY_HZ:.1f} Hz point, so this response "
          "was exercised by the injected control signal, exactly as a commissioning test should be.")



Window 1  08:32:52 to 08:37:26
  1. Stimulus applied: the controlling frequency was driven to 51.9 Hz, above the 50.5 Hz over-frequency point.
  2. Plant responded: active power fell from about 52 MW to a floor of 14 MW (26 percent of reference) while the frequency was high.
  3. Output recovered to its reference at 08:37:27, after the frequency dropped back below the point.

Window 2  09:04:35 to 09:07:37
  1. Stimulus applied: the controlling frequency was driven to 51.9 Hz, above the 50.5 Hz over-frequency point.
  2. Plant responded: active power fell from about 48 MW to a floor of 13 MW (28 percent of reference) while the frequency was high.
  3. Output recovered to its reference at 09:07:38, after the frequency dropped back below the point.

Measured grid frequency over the whole record stayed between 49.82 Hz and 50.21 Hz, never above the 50.5 Hz point, so this response was exercised by the injected control signal, exactly as a commissioning test should be.


### Checked against the South African Grid Code

The grid code requirements for renewable power plants [2] are more specific than the test record. Above the over-frequency point at 50.5 Hz the plant must reduce its active power along a straight line set by its droop, drawn as Figure 6 in the code. A droop of D percent means the output should fall all the way to zero over a frequency rise of D percent of the nominal frequency, so a smaller droop is a steeper cut. The plant must also trip if the frequency stays above 51.5 Hz for longer than four seconds. The cells below redraw each window against that required curve, measure how closely the reduction follows it, check the trip rule, and gather the grid-code verdict.

#### Figure 6: the required curve

Build the reduction curve the grid code requires, from the droop read from the sheet and the nominal frequency.


In [19]:
# =============================================================================
# FREQUENCY RESPONSE  -  the required Figure 6 curve (SAGC 6.1(2) and Figure 6, p.19)
#
# The grid code Figure 6 [2] sets the shape of the over-frequency reduction:
#   - at or below the over-frequency point the plant may run at full output (100%)
#   - above it the plant must reduce active power along a straight line whose steepness
#     is set by the droop. By the grid code's definition of droop (SAGC Section 4, p.8: a
#     percentage of the frequency change required to move from no-load to rated power), a
#     droop of D percent takes the output from full to zero over a frequency rise of D
#     percent of the nominal frequency.
# This cell builds that required curve from the droop read from the sheet (never a fixed
# value); the next cell draws each window against it.
# =============================================================================
import numpy as np                # builds the straight-line required curve
import matplotlib.pyplot as plt   # draws the figures
import matplotlib.dates as mdates # formats the time axis and the time colourbar

def required_power_percent(frequency, droop, f4, f_nominal):
    """The active power the Figure 6 curve allows, as a percent of the reference.

    At or below the over-frequency point f4 the full 100 percent is allowed. Above it
    the allowance falls along a straight line; a droop of `droop` percent uses up the
    whole 100 percent over a frequency rise of (droop / 100 * f_nominal) hertz. The
    result is held between 0 and 100 percent.
    """
    # full_swing_hz is the frequency rise that moves the plant across its whole output range.
    # It is exactly the grid code's definition of droop (SAGC Section 4, p.8): a droop of D
    # percent corresponds to D percent of the nominal frequency.
    full_swing_hz = (droop / 100.0) * f_nominal
    if full_swing_hz <= 0:
        return np.where(frequency > f4, 0.0, 100.0)
    allowed = 100.0 - (frequency - f4) / full_swing_hz * 100.0
    allowed = np.clip(allowed, 0.0, 100.0)
    return np.where(frequency <= f4, 100.0, allowed)


# Nominal frequency is read from the measured grid frequency (close to 50 Hz here).
nominal_freq = float(grid_freq.median())
print(f"Figure 6 curve ready: built from a {droop_percent:.0f} percent droop (read from the sheet) "
      f"and a {nominal_freq:.2f} Hz nominal, reducing above {F4_OVER_FREQUENCY_HZ:.1f} Hz.")


Figure 6 curve ready: built from a 4 percent droop (read from the sheet) and a 50.05 Hz nominal, reducing above 50.5 Hz.


#### Figure 6: draw each window against the curve

One figure per window: the measured reduction against the required curve (points coloured by time) and the same against the clock.


In [20]:
# =============================================================================
# FREQUENCY RESPONSE  -  draw each window against the Figure 6 curve
#
# Each window is shown two ways that share the same y axis (active power as a percent of
# the pre-threshold reference):
#   TOP    the shape against frequency, with the required Figure 6 curve laid on top, and
#          every measured point coloured by its clock time
#   BOTTOM the same percentages against the clock, the measured reduction and what the
#          curve required at each instant, so any point can be checked against the record.
# =============================================================================
def plot_figure6_window(window, n, total):
    control = scope[control_freq_col].loc[window["cross_up"]:window["cross_down"]]
    power = df[power_col].loc[window["cross_up"]:window["cross_down"]]
    reference_power = window["reference_power"]
    measured_percent = 100.0 * power / reference_power
    required_over_time = required_power_percent(control.values, droop_percent,
                                               F4_OVER_FREQUENCY_HZ, nominal_freq)

    fig, (ax_shape, ax_time) = plt.subplots(2, 1, figsize=(11, 11))

    # --- Top: the shape against frequency, with the required curve --------------
    freq_axis = np.linspace(F4_OVER_FREQUENCY_HZ - 0.6, control.max() + 0.3, 200)
    required_curve = required_power_percent(freq_axis, droop_percent, F4_OVER_FREQUENCY_HZ, nominal_freq)
    # Colour each measured point by its clock time so the reader can see when each dot
    # occurred; the colourbar on the right is keyed to the time of day, and it lines up
    # with the time panel below.
    point_times = mdates.date2num(control.index)
    scatter = ax_shape.scatter(control.values, measured_percent.values, s=16, c=point_times,
                               cmap="viridis", alpha=0.8,
                               label="measured active power (coloured by time)")
    time_bar = fig.colorbar(scatter, ax=ax_shape, pad=0.01)
    time_bar.set_label(f"time of day ({TIME_ZONE_LABEL})")
    time_bar.ax.yaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax_shape.plot(freq_axis, required_curve, color="#d62728", lw=2.2,
                  label=f"Figure 6 required curve ({droop_percent:.0f}% droop)")
    ax_shape.axvline(F4_OVER_FREQUENCY_HZ, color="#7f7f7f", ls=":", lw=1.3)
    ax_shape.annotate(f"over-frequency point {F4_OVER_FREQUENCY_HZ:.1f} Hz",
                      xy=(F4_OVER_FREQUENCY_HZ, 3), xytext=(5, 0), textcoords="offset points",
                      ha="left", va="bottom", fontsize=8.5, color="#7f7f7f")
    ax_shape.set_ylim(-5, 120)
    ax_shape.set_xlim(F4_OVER_FREQUENCY_HZ - 0.6, control.max() + 0.3)
    ax_shape.set_xlabel("Controlling frequency (Hz)")
    ax_shape.set_ylabel("Active power (% of reference)")
    ax_shape.set_title(f"{SITE_NAME} over-frequency reduction against Figure 6, window {n} of {total}, "
                       f"{window['cross_up']:%H:%M:%S} to {window['cross_down']:%H:%M:%S}")
    ax_shape.legend(loc="upper right", framealpha=0.9)

    # --- Bottom: the same percentages against the clock ------------------------
    # left axis: the power as a percent of reference (measured and what was required);
    # right axis: the controlling frequency that drove it, so stimulus and response sit
    # on one timeline.
    ax_time.plot(power.index, measured_percent.values, color="#1f77b4", lw=1.6,
                 label="measured active power")
    ax_time.plot(control.index, required_over_time, color="#d62728", lw=1.8, ls="--",
                 drawstyle="steps-post", label="power required by Figure 6")
    ax_time.set_ylim(0, 145)
    ax_time.set_ylabel("Active power (% of reference)")
    ax_time.set_xlabel(f"Time ({TIME_ZONE_LABEL})")
    ax_time.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax_time.set_title("the same reduction over time, for checking against the record")

    ax_freq_axis = ax_time.twinx()
    ax_freq_axis.plot(control.index, control.values, color="#9467bd", lw=1.8,
                      label="controlling frequency")
    ax_freq_axis.set_ylabel("Controlling frequency (Hz)", color="#9467bd")
    ax_freq_axis.tick_params(axis="y", labelcolor="#9467bd")
    # keep the frequency trace low in the panel so it stays clear of the power lines
    ax_freq_axis.set_ylim(control.min() - 0.3, control.max() + 4.0)

    moments = [
        (window["cross_up"], "#ff7f0e", f"{window['cross_up']:%H:%M:%S}"),
        (window["peak_time"], "#d62728", f"{window['peak_time']:%H:%M:%S}"),
        (window["cross_down"], "#2ca02c", f"{window['cross_down']:%H:%M:%S}"),
    ]
    mark_events(ax_time, moments, 100.0, gap=12, fontsize=8.2)

    # one combined legend covering both axes
    handles_left, labels_left = ax_time.get_legend_handles_labels()
    handles_right, labels_right = ax_freq_axis.get_legend_handles_labels()
    ax_time.legend(handles_left + handles_right, labels_left + labels_right,
                   loc="upper right", framealpha=0.9)

    fig.tight_layout()
    out_path = OUTPUT_DIR / f"{SITE_SLUG}_frequency_figure6_{window['cross_up']:%H%M%S}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Figure 6 comparison window {n} of {total}  figure {out_path.name}")
    plt.show()


# One Figure 6 comparison per window.
if not frequency_windows:
    print("No clean over-frequency window is present, so there is nothing to compare.")
for n, window in enumerate(frequency_windows, start=1):
    plot_figure6_window(window, n, len(frequency_windows))


Figure 6 comparison window 1 of 2  figure hartebeesthoek_frequency_figure6_083252.png


C:\Users\erick\AppData\Local\Temp\ipykernel_22136\3873474055.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figure 6 comparison window 2 of 2  figure hartebeesthoek_frequency_figure6_090435.png


### Reading the reduction against Figure 6

The top panel is the grid-code test in its usual form: active power as a percent of the pre-threshold reference up the vertical axis, the controlling frequency along the horizontal, and the red line the reduction that a 4 percent droop requires. A plant that followed the rule would have its points sitting on the red line. Here they do not. As soon as the frequency steps past 50.5 Hz the output collapses down a near-vertical column to a floor around 28 percent of reference and stays there, well below the red line, for every frequency up to the 51.9 Hz peak. The points only reach the line at the very top, where the required curve has itself fallen to that floor level.

The bottom panel says the same thing against the clock, which is the view to check against the record. The dashed red line is the power Figure 6 allows at each instant for the frequency shown on the right axis, so it steps down as the frequency steps up. The solid blue measured power should ride along that dashed line, but instead it drops straight to its floor at the first step and holds there, leaving a wide gap below the requirement for most of the window. The gap closes only briefly at the 51.9 Hz peak. In short the plant is not following the droop slope; it switches its output down to a fixed floor the moment the over-frequency point is crossed, more like an on-off curtailment than a proportional response. How far below the curve this sits, and what it means for the grid-code verdict, is measured in the cells that follow.

In [21]:
# =============================================================================
# FREQUENCY RESPONSE  -  does the reduction follow the Figure 6 curve?
#
# CHECK (reduction shape): while the frequency is above the over-frequency point the
# measured active power should sit ON the Figure 6 curve (SAGC 6.1(2), p.19), within the
# active-power control accuracy the grid code allows (about 2 percent, SAGC 6.2(11), p.21).
# We compare the measured percent to the required percent at every sample
# above the point and average the gap:
#   gap = required percent - measured percent
#   gap well above zero  -> the plant cut TOO MUCH (over-reduced)
#   gap well below zero  -> the plant cut TOO LITTLE (under-reduced)
#   gap near zero        -> the plant followed the curve
# =============================================================================

if not frequency_windows:
    print("No over-frequency window to compare against the Figure 6 curve.")

for n, window in enumerate(frequency_windows, start=1):
    control = scope[control_freq_col].loc[window["cross_up"]:window["cross_down"]]
    power = df[power_col].loc[window["cross_up"]:window["cross_down"]]
    reference_power = window["reference_power"]

    measured_percent = 100.0 * power.values / reference_power
    required_percent = required_power_percent(control.values, droop_percent,
                                              F4_OVER_FREQUENCY_HZ, nominal_freq)
    above_point = control.values > F4_OVER_FREQUENCY_HZ
    gap = required_percent[above_point] - measured_percent[above_point]
    average_gap = float(gap.mean()) if gap.size else float("nan")

    # The acceptance decision for this window.
    if average_gap > ACTIVE_POWER_ACCURACY_PERCENT:
        shape_pass = False
        verdict = (f"FAIL, over-reduced: the output sat on average {average_gap:.0f} percentage points "
                   f"BELOW the curve, far outside the {ACTIVE_POWER_ACCURACY_PERCENT:.0f} percent the grid "
                   "code allows.")
    elif average_gap < -ACTIVE_POWER_ACCURACY_PERCENT:
        shape_pass = False
        verdict = (f"FAIL, under-reduced: the output sat on average {abs(average_gap):.0f} percentage points "
                   f"ABOVE the curve, far outside the {ACTIVE_POWER_ACCURACY_PERCENT:.0f} percent allowed.")
    else:
        shape_pass = True
        verdict = (f"PASS: the output stayed within {ACTIVE_POWER_ACCURACY_PERCENT:.0f} percentage points of the "
                   "curve, so it followed the required droop.")

    # Keep the result on the window so the combined findings can read it.
    window["shape_average_gap"] = average_gap
    window["shape_pass"] = shape_pass

    print(f"Window {n}  {window['cross_up']:%H:%M:%S} to {window['cross_down']:%H:%M:%S}")
    print(f"  Reduction shape check: {verdict}")


Window 1  08:32:52 to 08:37:26
  Reduction shape check: FAIL, over-reduced: the output sat on average 24 percentage points BELOW the curve, far outside the 2 percent the grid code allows.
Window 2  09:04:35 to 09:07:37
  Reduction shape check: FAIL, over-reduced: the output sat on average 26 percentage points BELOW the curve, far outside the 2 percent the grid code allows.


In [22]:
# =============================================================================
# FREQUENCY RESPONSE  -  the trip requirement above 51.5 Hz
#
# CHECK (trip): the grid code says that once the grid frequency stays above 51.5 Hz for
# longer than 4 seconds the plant must trip (disconnect) to protect the network [2]
# (SAGC 6.1(3), p.19). For
# each window we measure how long the controlling frequency stayed above 51.5 Hz and
# whether the plant actually tripped (output fell to zero). The honest distinction
# between the injected control frequency and the measured grid frequency is kept in view:
# the trip protection is meant to watch the true grid frequency.
# =============================================================================

# FREQ_TRIP_OUTPUT_PERCENT (config) is the output, as a percent of reference, at or below
# which the plant is read as having disconnected. A notebook reading aid, not a grid-code
# figure; the code only says the plant must trip.

def longest_run_seconds(flag_above):
    """Longest continuous stretch, in seconds, where a boolean series stays True (here,
    while the frequency is above the trip point)."""
    longest = pd.Timedelta(0)
    run_start = None
    previous_time = None
    for timestamp, is_above in flag_above.items():
        if is_above and run_start is None:
            run_start = timestamp
        if (not is_above) and run_start is not None:
            longest = max(longest, previous_time - run_start)
            run_start = None
        previous_time = timestamp
    if run_start is not None:
        longest = max(longest, flag_above.index[-1] - run_start)
    return longest.total_seconds()


if not frequency_windows:
    print("No over-frequency window, so the trip requirement is not exercised.")

for n, window in enumerate(frequency_windows, start=1):
    control = scope[control_freq_col].loc[window["cross_up"]:window["cross_down"]]
    power = df[power_col].loc[window["cross_up"]:window["cross_down"]]
    reference_power = window["reference_power"]

    above_trip_point = control > F5_TRIP_HZ
    seconds_above_trip = longest_run_seconds(above_trip_point)
    trip_demanded_by_signal = seconds_above_trip > TRIP_HOLD_SECONDS

    lowest_percent = 100.0 * float(power.min()) / reference_power if reference_power else float("nan")
    plant_tripped = lowest_percent < FREQ_TRIP_OUTPUT_PERCENT   # output essentially at zero means a disconnect

    # Keep the result on the window so the combined findings can read it.
    window["trip_demanded_by_signal"] = trip_demanded_by_signal
    window["plant_tripped"] = plant_tripped

    print(f"Window {n}  {window['cross_up']:%H:%M:%S} to {window['cross_down']:%H:%M:%S}")
    print(f"  Controlling frequency stayed above {F5_TRIP_HZ:.1f} Hz for about {seconds_above_trip:.0f} s "
          f"(a trip is required after {TRIP_HOLD_SECONDS:.0f} s above it).")
    if plant_tripped:
        print(f"  The plant output fell to about {lowest_percent:.0f} percent of reference, so it disconnected.")
    else:
        print(f"  The plant stayed connected: output held near {lowest_percent:.0f} percent of reference, "
              "not zero.")

# The honest distinction between the test signal and the real grid frequency.
if frequency_windows:
    print(f"\nNote: the {F5_TRIP_HZ:.1f} Hz reached here is the INJECTED control frequency. The measured grid "
          f"frequency stayed between {grid_freq.min():.2f} Hz and {grid_freq.max():.2f} Hz, never near "
          f"{F5_TRIP_HZ:.1f} Hz, so no real trip was called for. Staying connected is the correct outcome, "
          "because the trip protection keys on the true grid frequency, not on the injected test signal.")


Window 1  08:32:52 to 08:37:26
  Controlling frequency stayed above 51.5 Hz for about 42 s (a trip is required after 4 s above it).
  The plant stayed connected: output held near 26 percent of reference, not zero.
Window 2  09:04:35 to 09:07:37
  Controlling frequency stayed above 51.5 Hz for about 30 s (a trip is required after 4 s above it).
  The plant stayed connected: output held near 28 percent of reference, not zero.

Note: the 51.5 Hz reached here is the INJECTED control frequency. The measured grid frequency stayed between 49.82 Hz and 50.21 Hz, never near 51.5 Hz, so no real trip was called for. Staying connected is the correct outcome, because the trip protection keys on the true grid frequency, not on the injected test signal.


In [23]:
# =============================================================================
# FREQUENCY RESPONSE  -  the grid code verdict
#
# This pulls the grid-code checks together into one honest verdict, per window and for
# the test as a whole:
#   - the reduction shape against the Figure 6 curve (from the shape check)
#   - the trip behaviour above 51.5 Hz (from the trip check)
# The verdict is stated plainly and constructively: what works, what does not, and the
# next step.
# =============================================================================

if not frequency_windows:
    print("No over-frequency window was captured, so there is no grid-code verdict to give.")

all_windows_over_reduced = True
for n, window in enumerate(frequency_windows, start=1):
    average_gap = window.get("shape_average_gap", float("nan"))
    shape_pass = window.get("shape_pass", False)
    if shape_pass:
        all_windows_over_reduced = False

    print(f"Window {n}  {window['cross_up']:%H:%M:%S} to {window['cross_down']:%H:%M:%S}")
    if shape_pass:
        print("  Figure 6 shape: PASS, the reduction followed the required droop curve.")
    else:
        print(f"  Figure 6 shape: FAIL, the plant over-reduced, sitting about {average_gap:.0f} percentage "
              "points below the required curve.")
    if window.get("plant_tripped"):
        print("  Trip: the plant disconnected during this window.")
    else:
        print("  Trip: the plant stayed connected, which is correct since the real grid frequency never "
              "approached the 51.5 Hz trip point.")

# The overall, constructive verdict.
print("\nOverall grid-code verdict:")
if frequency_windows and all_windows_over_reduced:
    print("  The over-frequency response FAILS the Figure 6 requirement, but only on the SIZE of the cut, "
          "not on its presence. The response is real and in the right direction: every time the frequency "
          "rose past 50.5 Hz the plant reduced power, and it recovered when the frequency fell back. The "
          "problem is that it cuts too far, dropping straight to a fixed floor near a quarter of output "
          "instead of following the gentler droop slope, so it behaves like an on-off curtailment.")
    print("  Next step: retune the droop so the reduction tracks the Figure 6 slope rather than flooring, "
          "then repeat the test. The trip protection behaved correctly and needs no change.")
elif frequency_windows:
    print("  The over-frequency response meets the Figure 6 requirement in at least one window; see the "
          "per-window lines above for the detail.")


Window 1  08:32:52 to 08:37:26
  Figure 6 shape: FAIL, the plant over-reduced, sitting about 24 percentage points below the required curve.
  Trip: the plant stayed connected, which is correct since the real grid frequency never approached the 51.5 Hz trip point.
Window 2  09:04:35 to 09:07:37
  Figure 6 shape: FAIL, the plant over-reduced, sitting about 26 percentage points below the required curve.
  Trip: the plant stayed connected, which is correct since the real grid frequency never approached the 51.5 Hz trip point.

Overall grid-code verdict:
  The over-frequency response FAILS the Figure 6 requirement, but only on the SIZE of the cut, not on its presence. The response is real and in the right direction: every time the frequency rose past 50.5 Hz the plant reduced power, and it recovered when the frequency fell back. The problem is that it cuts too far, dropping straight to a fixed floor near a quarter of output instead of following the gentler droop slope, so it behaves like an

## Delta production constraint test

The delta production constraint asks the plant to hold back a slice of the power it could be producing, so the held-back margin is available as a reserve the grid operator can call on. The slice is set as a percentage of the plant's **available power (Pavailable)**, the most it could generate at that moment from its wind or sun. A delta of 10 percent means the plant runs at 90 percent of what it could, keeping 10 percent in hand. The South African Grid Code calls this PDelta and defines it as the amount of active power by which the available power has been reduced to provide reserves for frequency stabilisation [2].

The procedure [1] sends a delta setpoint, switches the mode on and checks the output drops by that percentage, then switches off and checks it returns to full output. The grid code adds three requirements [2]: the reduction must match the command within about 2 percent, it must commence within 2 seconds and complete within 10, and the plant must manage a delta of at least 3 percent. This notebook assesses the accuracy of the reduction; the timing and minimum-capability checks need a logged order time and an available-power signal this record lacks, so they are noted but not assessed.

This logger has no available-power channel, so available power is taken as the output just before delta mode comes on. Because that level can drift on a renewable plant, a reduction that misses the commanded percentage is reported with this caveat rather than read as a hard pass or fail.

In [24]:
# =============================================================================
# DELTA PRODUCTION CONSTRAINT  -  reading the test off the data (basic facts)
#
# Before measuring anything we read the plain facts of this test:
#   - the delta mode on/off flag
#   - the delta setpoint channel and the percentages it was set to
#   - whether the logger records the available power directly (if not, it is inferred
#     from the output just before delta mode comes on)
# =============================================================================

# 1. The channels this test needs, found by role (never by a fixed column name).
delta_mode_col = resolve("delta_mode", required=True)   # delta mode on/off flag
delta_sp_col   = resolve("delta_sp", required=True)     # delta setpoint (percent of available)
power_col      = resolve("poc_p", required=True)        # measured active power at the POC
available_col  = resolve("available")                   # available power, if the sheet records it

# 2. The delta percentages commanded while the mode was on.
delta_on = df[delta_mode_col].astype(bool)
setpoints_while_on = df.loc[delta_on, delta_sp_col].dropna().unique()
print(f"Delta mode is on for {int(delta_on.sum())} samples")
print(f"Delta setpoints commanded while on: "
      f"{sorted(float(v) for v in setpoints_while_on)} percent of available power")

# 3. Is the available power recorded directly?
if available_col is not None:
    print(f"Available power channel: {available_col}")
else:
    print("No available-power channel in this record, so available power is inferred from the "
          "output measured just before delta mode comes on.")


Delta mode is on for 162 samples
Delta setpoints commanded while on: [10.0, 15.0] percent of available power
No available-power channel in this record, so available power is inferred from the output measured just before delta mode comes on.


In [25]:
# =============================================================================
# DELTA PRODUCTION CONSTRAINT  -  finding the delta windows
#
# A "window" is one occurrence of the test (one delta-mode-on period). We keep the clean
# windows, those with NO other active-power constraint mode on (curtailment or power
# gradient), so the reduction we measure is the delta reserve and nothing else. For each
# window we measure, straight from the data:
#   - the commanded delta setpoint (percent of available power)
#   - the available power (from its channel if present, otherwise the output just before
#     mode on)
#   - the target output the delta implies: available * (1 - delta / 100)
#   - the settled output held while the mode was on
#   - the actual reduction delivered, as a percent of available
#   - the setpoint-sent, mode-on and mode-off moments, and the recovery after mode off
# =============================================================================

# 1. Other active-power constraint modes; a window overlapping them is set aside.
ap_flag = resolve("ap_mode")
pg_flag = resolve("pg_mode")
other_constraints = [f for f in [ap_flag, pg_flag] if f]

# 2. Restrict to the configured scope (whole record unless limited in the config cell).
delta_cfg = EVENT_WINDOWS.get("delta")
delta_region = df.loc[delta_cfg[0]:delta_cfg[1]] if (delta_cfg and all(delta_cfg)) else df


def infer_available_power(mode_on, mode_off):
    """The available power for this window. Read from its own channel if the sheet has
    one, otherwise taken as the output just before delta mode came on.
    """
    if available_col is not None:
        return float(df[available_col].loc[mode_on:mode_off].median())
    before = df[power_col].loc[mode_on - pd.Timedelta(seconds=DELTA_AVAILABLE_LOOKBACK_SECONDS):mode_on]
    before = before[before.index < mode_on]
    return float(before.median()) if len(before) else float(df[power_col].loc[mode_on])


def settled_output(mode_on, mode_off):
    """The output the plant settled at while the mode was on, taken as the median over the
    last part of the on-window so the initial ramp down is not counted.
    """
    on_slice = df[power_col].loc[mode_on:mode_off]
    tail = on_slice.loc[mode_off - pd.Timedelta(seconds=DELTA_SETTLED_TAIL_SECONDS):mode_off]
    return float(tail.median()) if len(tail) else float(on_slice.median())


# 3. Find and measure each clean window.
delta_windows = []
for mode_on, mode_off in on_segments(delta_region[delta_mode_col]):
    overlaps = any(delta_region[f].astype(bool).loc[mode_on:mode_off].mean() >= 0.5
                   for f in other_constraints)
    if overlaps:
        continue

    setpoint_series = df[delta_sp_col]
    setpoint = float(setpoint_series.loc[mode_on:mode_off].dropna().iloc[0])

    # the setpoint-sent moment: the last change to the setpoint at or before mode on
    setpoint_changes = setpoint_series.ne(setpoint_series.shift())
    setpoint_changes.iloc[0] = False
    sent_before = setpoint_series.index[setpoint_changes & (setpoint_series.index <= mode_on)]
    setpoint_sent = sent_before[-1] if len(sent_before) else None

    available_power = infer_available_power(mode_on, mode_off)
    target_output = available_power * (1 - setpoint / 100.0)
    settled = settled_output(mode_on, mode_off)
    actual_reduction_pct = (100.0 * (available_power - settled) / available_power
                            if available_power else float("nan"))

    after = df[power_col].loc[mode_off:]
    after = after[after.index > mode_off]
    recovery_output = float(after.head(DELTA_RECOVERY_SAMPLES).median()) if len(after) else None

    # accuracy check: how the measured reduction compares to the commanded delta, within the
    # grid code active-power accuracy (against inferred available power when no channel exists).
    accuracy_gap = actual_reduction_pct - setpoint if pd.notna(actual_reduction_pct) else float("nan")
    within_band = bool(pd.notna(accuracy_gap) and abs(accuracy_gap) <= ACTIVE_POWER_ACCURACY_PERCENT)

    delta_windows.append({
        "mode_on": mode_on,
        "mode_off": mode_off,
        "setpoint": setpoint,
        "setpoint_sent": setpoint_sent,
        "available_power": available_power,
        "target_output": target_output,
        "settled_output": settled,
        "actual_reduction_pct": actual_reduction_pct,
        "recovery_output": recovery_output,
        "accuracy_gap": accuracy_gap,
        "within_band": within_band,
    })

# 4. Report what was found.
print(f"Clean delta windows found: {len(delta_windows)}")
for n, w in enumerate(delta_windows, start=1):
    print(f"  Window {n}: {w['mode_on']:%H:%M:%S} to {w['mode_off']:%H:%M:%S}, "
          f"setpoint {w['setpoint']:.0f}%, available ~{w['available_power']:.1f} MW, "
          f"target ~{w['target_output']:.1f} MW, settled ~{w['settled_output']:.1f} MW, "
          f"actual reduction {w['actual_reduction_pct']:.1f}%")
if not delta_windows:
    print("Delta mode is never on without another constraint here, so there is nothing to plot.")


Clean delta windows found: 2
  Window 1: 08:26:14 to 08:27:36, setpoint 10%, available ~47.7 MW, target ~42.9 MW, settled ~45.9 MW, actual reduction 3.7%
  Window 2: 09:00:45 to 09:02:05, setpoint 15%, available ~50.2 MW, target ~42.7 MW, settled ~42.4 MW, actual reduction 15.5%


In [26]:
# =============================================================================
# DELTA PRODUCTION CONSTRAINT  -  one graph per delta window
#
# For each window we draw the measured active power together with two reference levels:
#   - the available power (the full output just before the mode came on)
#   - the delta target, which is (100 - delta) percent of available, the level the plant
#     should settle at while holding the reserve
# The setpoint-sent, mode-on and mode-off moments are marked with their times. The reader
# should be able to see the output step down towards the target and recover at mode off,
# and read off whether it actually reached the target.
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_delta_window(w, n, total):
    win = window_around(w["mode_on"], w["mode_off"])   # padded slice for context
    power = win[power_col]
    available = w["available_power"]
    target = w["target_output"]
    setpoint = w["setpoint"]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(power.index, power, color="#1f77b4", lw=1.6, label="active power (measured)")
    ax.axhline(available, color="#7f7f7f", ls="--", lw=1.3,
               label=f"available power (inferred) {available:.0f} MW")
    ax.axhline(target, color="#d62728", ls="--", lw=1.6,
               label=f"delta target, {100 - setpoint:.0f}% of available = {target:.0f} MW")

    # mark the procedure moments with their times
    events = []
    if (w["setpoint_sent"] is not None
            and abs(w["setpoint_sent"] - w["mode_on"]) > pd.Timedelta(seconds=2)):
        events.append((w["setpoint_sent"], "#7f7f7f",
                       f"{w['setpoint_sent']:%H:%M:%S}\nsetpoint {setpoint:.0f}% sent\n(mode still off)"))
    events.append((w["mode_on"], "#ff7f0e",
                   f"{w['mode_on']:%H:%M:%S}\ndelta mode ON\n(output should drop {setpoint:.0f}%)"))
    events.append((w["mode_off"], "#2ca02c",
                   f"{w['mode_off']:%H:%M:%S}\ndelta mode OFF\n(back to full output)"))
    mark_events(ax, events, power.max(), gap=6, fontsize=8.2)

    span = power.max() - power.min()
    ax.set_ylim(power.min() - max(4, 0.2 * span), power.max() + max(16, 0.9 * span))
    ax.set_xlabel(f"Time ({TIME_ZONE_LABEL})")
    ax.set_ylabel("Active power (MW)")
    ax.set_title(f"{SITE_NAME} delta production constraint window {n} of {total}, "
                 f"{setpoint:.0f}% delta at {w['mode_on']:%H:%M:%S}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.legend(loc="upper right", framealpha=0.9)

    out_path = OUTPUT_DIR / f"{SITE_SLUG}_delta_{w['mode_on']:%H%M%S}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    # --- Findings, written straight from the checks --------------------------
    inferred = available_col is None
    inference_note = (" (available power inferred from the output just before mode on, so this "
                      "reading is indicative)") if inferred else ""
    story = []
    if w["within_band"]:                                                       # accuracy check
        story.append(f"Accuracy check pass: the output settled about {w['actual_reduction_pct']:.1f}% below "
                     f"available, within the +/-{ACTIVE_POWER_ACCURACY_PERCENT:.0f}% accuracy of the "
                     f"{w['setpoint']:.0f}% commanded{inference_note}.")
    elif pd.isna(w["accuracy_gap"]):
        story.append("Accuracy check not assessable: the available power could not be read for this window.")
    elif w["accuracy_gap"] < 0:
        story.append(f"Accuracy check under-delivered: the output reduced only about {w['actual_reduction_pct']:.1f}% "
                     f"against the {w['setpoint']:.0f}% commanded, outside the +/-{ACTIVE_POWER_ACCURACY_PERCENT:.0f}% "
                     f"accuracy{inference_note}.")
    else:
        story.append(f"Accuracy check over-delivered: the output reduced about {w['actual_reduction_pct']:.1f}% "
                     f"against the {w['setpoint']:.0f}% commanded, outside the +/-{ACTIVE_POWER_ACCURACY_PERCENT:.0f}% "
                     f"accuracy{inference_note}.")
    if w["recovery_output"] is not None:
        story.append(f"After mode off the output recovered to about {w['recovery_output']:.0f} MW.")
    else:
        story.append("The recovery after mode off is not captured before the data ends in this window.")

    print(f"\nDelta window {n} of {total}  {w['setpoint']:.0f}% delta  figure {out_path.name}")
    for k, line in enumerate(story, start=1):
        print(f"  {k}. {line}")
    plt.show()


# One graph per window.
if not delta_windows:
    print("No clean delta window to plot.")
for n, w in enumerate(delta_windows, start=1):
    plot_delta_window(w, n, len(delta_windows))



Delta window 1 of 2  10% delta  figure hartebeesthoek_delta_082614.png
  1. Accuracy check under-delivered: the output reduced only about 3.7% against the 10% commanded, outside the +/-2% accuracy (available power inferred from the output just before mode on, so this reading is indicative).
  2. After mode off the output recovered to about 49 MW.


C:\Users\erick\AppData\Local\Temp\ipykernel_22136\3923486939.py:81: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Delta window 2 of 2  15% delta  figure hartebeesthoek_delta_090045.png
  1. Accuracy check pass: the output settled about 15.5% below available, within the +/-2% accuracy of the 15% commanded (available power inferred from the output just before mode on, so this reading is indicative).
  2. After mode off the output recovered to about 47 MW.


### What the delta windows show

The two windows tell different halves of the same story, and together they show the delta reserve working but not yet holding consistently.

The second window behaves exactly as the procedure expects. While the 15 percent setpoint was sent at 09:00:19 the output kept running near its available level of about 50 MW, so the command was received without acting on it. The moment delta mode came on at 09:00:45 the output stepped down promptly and settled right on the 43 MW target line, a measured reduction of about 15.5 percent against the 15 percent commanded, inside the grid code accuracy of plus or minus 2 percent [2]. It held there for the full window and sprang back to its available output the instant the mode went off at 09:02:05. This is a clean pass: the plant found and held very close to the reserve it was asked to hold.

The first window is where the plant falls short. The 10 percent setpoint behaved correctly at the edges, inert while sent at 08:25:48 and recovering at mode off at 08:27:36, and the output did dip down to the 43 MW target just after mode on. But it did not stay there. Within half a minute it drifted back up and settled around 46 MW, a reduction of only about 3.7 percent against the 10 percent commanded, well short of the target and outside the accuracy band. The reserve was established briefly and then largely given back.

Two things temper how hard to read the first window. The available power here is inferred from the output measured just before delta mode comes on, because this record carries no available-power channel, so if the wind or sun changed during the window the true available level moved with it and the measured percentage is only approximate. And the plant clearly has the mechanism, since the 15 percent window proves it can find and hold a target accurately. The honest verdict is that delta control is present and works in the larger window but under-delivered in the smaller one, and the natural next step is to confirm whether the drift in the 10 percent window was the resource changing or the controller relaxing the reduction, ideally with an available-power signal logged so the reserve can be judged without inference.

## Voltage, reactive power and power factor modes

These three modes are the plant's reactive-side controls, and the grid operator runs them one at a time. **Reactive power**, measured in **megavolt-amperes reactive (MVAr)**, is the part of the electrical flow that does no real work but sets the voltage at the **point of connection (POC)**, the spot where the plant meets the grid. Pushing reactive power out (injecting) lifts the local voltage; pulling it in (absorbing) lowers it.

**Voltage control** is the mode where the plant is given a target voltage in kilovolts (kV) and trims its reactive power up or down to hold the POC at that target. The **droop** sets how stiff that response is: the South African Grid Code defines voltage droop as the per-unit voltage change caused by a per-unit change in reactive power [2], so a smaller droop percentage means a stiffer hold. A higher target asks the plant to inject reactive power, a lower target to absorb it.

**Reactive power (Q) mode** instead commands the reactive power directly to a setpoint in MVAr, holding it whatever the voltage does. **Power factor (PF) mode** commands the **power factor (PF)**, the ratio of real power to total power, a number between -1 and +1 where unity (1.0) means no reactive power at all; the plant then supplies reactive power in proportion to its real output to keep that ratio. The three modes are mutually exclusive, so only one is on at a time [2].

The acceptance procedure [1] exercises each mode in turn, stepping its setpoint up and down and, for voltage mode, changing the droop, while checking the measured channel follows. The cells below read each mode from the data: where a mode and its steps are present they are drawn and assessed, and where a mode was not switched on during the capture that is reported plainly rather than assumed.

### Voltage mode

This subsection reads the voltage-control test. It finds every period where voltage mode is on, and within each it looks for the procedure steps: a change in the voltage setpoint (a higher or lower target) and a change in the droop setting. For each setpoint step it checks that reactive power was injected for a higher target or absorbed for a lower one. Where the setpoint and droop are simply held without stepping, that is reported as a held baseline, with a plain statement that the stepped part of the test was not exercised in this capture. The graph shows the measured voltage against its setpoint with the droop labelled on the upper panel, and the reactive power against its zero reference on the lower panel, with the active-power constraint tests shaded so their effect on reactive power is visible.

In [27]:
# =============================================================================
# VOLTAGE MODE TEST
#
# What the plant must do, from the acceptance procedure [1] and grid code [2]:
#   CHECK 1  Voltage mode ON with a reference voltage set -> the measured POC voltage
#            should sit at that reference, within the grid-code accuracy band.
#   CHECK 2  Setpoint stepped to a HIGHER target -> voltage should rise towards it and
#            reactive power should be INJECTED (positive MVAr).
#   CHECK 3  Setpoint stepped to a LOWER target -> voltage should fall towards it and
#            reactive power should be ABSORBED (negative MVAr).
#   CHECK 4  Droop setting changed (for example 4% to 8%) -> the change is confirmed.
#
# A "window" is one period of voltage mode being on. We find every window, and for each
# we read whether the setpoint and droop actually stepped. Where they were only held, the
# baseline is reported honestly and the stepped checks are marked not exercised, never
# faked. Every value is read from the data.
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. The channels this test needs, found by role (never by a fixed column name).
v_mode_col     = resolve("v_mode", required=True)   # voltage mode on/off flag
voltage_col    = resolve("v_meas", required=True)   # measured POC voltage (kV)
v_setpoint_col = resolve("v_sp", required=True)     # voltage setpoint (kV)
reactive_col   = resolve("q_meas", required=True)   # measured reactive power (MVAr)
droop_col      = resolve("droop_v")                 # voltage droop setting (percent), if recorded

# active-power constraint flags, shaded on the plot so their effect on reactive power shows
active_power_flags = [resolve(r) for r in ["ap_mode", "pg_mode", "delta_mode"]]
active_power_flags = [flag for flag in active_power_flags if flag]

# 2. Restrict to the configured scope (whole record unless limited in the config cell).
v_cfg = EVENT_WINDOWS.get("voltage")
v_region = df.loc[v_cfg[0]:v_cfg[1]] if (v_cfg and all(v_cfg)) else df


def changes_in(series):
    """The moments a held value steps to a new one, as (time, new value) pairs. The first
    sample is the window edge, not a change, so it is never counted."""
    stepped = series.ne(series.shift())
    if len(stepped):
        stepped.iloc[0] = False
    return [(ts, float(series.loc[ts])) for ts in series.index[stepped] if pd.notna(series.loc[ts])]


def shade_active_power(ax, win):
    """Lightly shade the periods where any active-power constraint test was on, so the
    reader can see why reactive power moves during those stretches."""
    labelled = False
    for flag in active_power_flags:
        for seg_start, seg_end in on_segments(win[flag]):
            ax.axvspan(seg_start, seg_end, color="#999999", alpha=0.12,
                       label="active-power test on" if not labelled else None)
            labelled = True


def plot_voltage_window(mode_on, mode_off, n, total):
    win      = window_around(mode_on, mode_off)          # padded slice for context
    voltage  = win[voltage_col]
    reactive = win[reactive_col]
    setpoint = win[v_setpoint_col]

    # --- Read the procedure steps straight from the data --------------------
    on_setpoint = df[v_setpoint_col].loc[mode_on:mode_off]
    on_reactive = df[reactive_col].loc[mode_on:mode_off]
    on_voltage  = df[voltage_col].loc[mode_on:mode_off]
    reference   = float(on_setpoint.dropna().iloc[0]) if on_setpoint.notna().any() else float("nan")
    setpoint_steps = changes_in(on_setpoint)
    droop_on    = df[droop_col].loc[mode_on:mode_off] if droop_col else None
    droop_steps = changes_in(droop_on) if droop_col else []
    droop_held  = float(droop_on.dropna().iloc[0]) if (droop_col and droop_on.notna().any()) else None

    # --- CHECK 1: is the measured voltage held at its reference? -------------
    # Guard a missing or zero reference: with no usable setpoint there is no band to judge
    # against, so the hold check is reported as not performed rather than read as a fail.
    reference_usable = pd.notna(reference) and reference != 0
    if reference_usable:
        accuracy_band    = abs(reference) * VOLTAGE_ACCURACY_PERCENT / 100.0
        within_band      = on_voltage.sub(reference).abs() <= accuracy_band
        fraction_in_band = float(within_band.mean()) if len(within_band) else 0.0
        CHECK1_held      = fraction_in_band >= VOLTAGE_HELD_FRACTION   # held at reference for most of the window
    else:
        fraction_in_band = 0.0
        CHECK1_held      = False
    median_voltage   = float(on_voltage.median())
    median_reactive  = float(on_reactive.median())

    # --- Draw the two-panel graph -------------------------------------------
    fig, (ax_v, ax_q) = plt.subplots(2, 1, figsize=(12, 9.5), sharex=True,
                                     gridspec_kw={"height_ratios": [3, 2]})
    shade_active_power(ax_v, win)
    shade_active_power(ax_q, win)

    # upper panel: measured voltage against its setpoint, with the droop labelled
    ax_v.plot(voltage.index, voltage, color="#1f77b4", lw=1.6, label="POC voltage (measured)")
    ax_v.plot(setpoint.index, setpoint, color="#d62728", lw=1.8, ls="--", drawstyle="steps-post",
              label="voltage setpoint")
    if pd.notna(reference):
        droop_text = f", {droop_held:.0f}% droop" if droop_held is not None else ""
        ax_v.annotate(f"setpoint {reference:.0f} kV{droop_text}", xy=(setpoint.index[0], reference),
                      xytext=(4, 4), textcoords="offset points", ha="left", va="bottom",
                      fontsize=8, color="#d62728",
                      bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))
    ax_v.set_ylabel("Voltage (kV)")

    # lower panel: reactive power against its zero reference
    ax_q.plot(reactive.index, reactive, color="#9467bd", lw=1.6, label="reactive power (measured)")
    ax_q.axhline(0.0, color="#7f7f7f", ls="--", lw=1.2, label="zero reactive power")
    ax_q.set_ylabel("Reactive power (MVAr)")

    # generous headroom so the legend and the lifted time labels never touch the traces
    v_span = (voltage.max() - voltage.min()) or 1.0
    ax_v.set_ylim(voltage.min() - 0.3 * v_span, voltage.max() + 1.4 * v_span)
    q_span = (reactive.max() - reactive.min()) or 1.0
    ax_q.set_ylim(reactive.min() - 0.3 * q_span, reactive.max() + 0.5 * q_span)

    # mark the mode-on, mode-off, setpoint-step and droop-step moments with their times
    events = [(mode_on, "#ff7f0e", f"{mode_on:%H:%M:%S}\nvoltage mode ON")]
    for ts, value in setpoint_steps:
        events.append((ts, "#d62728", f"{ts:%H:%M:%S}\nsetpoint {value:.0f} kV"))
    for ts, value in droop_steps:
        events.append((ts, "#8c564b", f"{ts:%H:%M:%S}\ndroop {value:.0f}%"))
    events.append((mode_off, "#2ca02c", f"{mode_off:%H:%M:%S}\nvoltage mode OFF"))
    levels = mark_steps(ax_v, events)

    ax_v.set_title(f"{SITE_NAME} voltage mode window {n} of {total}, reference {reference:.0f} kV "
                   f"at {mode_on:%H:%M:%S}", pad=title_pad_for(levels))
    ax_q.set_xlabel(f"Time ({TIME_ZONE_LABEL})")
    ax_q.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax_v.legend(loc="upper right", framealpha=0.9)
    ax_q.legend(loc="upper right", framealpha=0.9)
    out_path = OUTPUT_DIR / f"{SITE_SLUG}_voltage_{mode_on:%H%M%S}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    # --- Findings, written straight from the checks above -------------------
    story = []
    if not reference_usable:                                                   # CHECK 1
        story.append(f"Check 1 not performed: no usable voltage reference was recorded while the "
                     f"mode was on, so the measured POC voltage (near {median_voltage:.1f} kV) cannot "
                     f"be judged against a reference in this window.")
    elif CHECK1_held:
        story.append(f"Check 1 pass: while voltage mode was on the measured POC voltage held near "
                     f"{median_voltage:.1f} kV, within {VOLTAGE_ACCURACY_PERCENT:.1f}% of the "
                     f"{reference:.0f} kV reference for {fraction_in_band * 100:.0f}% of the window.")
    else:
        story.append(f"Check 1: the measured POC voltage sat near {median_voltage:.1f} kV against a "
                     f"{reference:.0f} kV reference, inside the {VOLTAGE_ACCURACY_PERCENT:.1f}% band for "
                     f"{fraction_in_band * 100:.0f}% of the window, so it was held close to but not "
                     f"continuously at the reference (the active-power tests move it).")
    if droop_held is not None:
        story.append(f"The droop was set to {droop_held:.0f}% for this window.")
    if setpoint_steps:                                                         # CHECK 2 and 3
        # Each step is judged against the setpoint just before it (not the window's first
        # reference), and by how the reactive power CHANGED across the step. An up-step should
        # raise reactive power (more injection), a down-step should lower it (more absorption).
        # The before and after readings are capped to the neighbouring steps so a nearby step
        # does not bleed into the reading.
        step_times = [ts for ts, _ in setpoint_steps]
        window_seconds = pd.Timedelta(seconds=VOLTAGE_STEP_REACTIVE_SECONDS)
        for i, (ts, value) in enumerate(setpoint_steps):
            previous_setpoint = reference if i == 0 else setpoint_steps[i - 1][1]
            previous_step_time = mode_on if i == 0 else step_times[i - 1]
            next_step_time = step_times[i + 1] if i + 1 < len(step_times) else mode_off

            before_start = max(previous_step_time, ts - window_seconds)
            before = on_reactive.loc[before_start:ts]
            before = before[before.index < ts]
            after_end = min(ts + window_seconds, next_step_time)
            after = on_reactive.loc[ts:after_end]
            after = after[after.index > ts]
            reactive_before = float(before.median()) if len(before) else float("nan")
            reactive_after = float(after.median()) if len(after) else float("nan")
            change = reactive_after - reactive_before

            raised   = value > previous_setpoint
            check_no = 2 if raised else 3
            verb     = "higher" if raised else "lower"
            expected = "rise" if raised else "fall"
            if change > VOLTAGE_REACTIVE_DEADBAND_MVAR:
                moved = "rose"
            elif change < -VOLTAGE_REACTIVE_DEADBAND_MVAR:
                moved = "fell"
            else:
                moved = "held"
            expected_moved = "rose" if raised else "fell"
            if pd.isna(change):
                tag = "not captured"
            else:
                tag = "pass" if moved == expected_moved else "fail"
            story.append(f"Check {check_no} {tag}: setpoint stepped {verb} from {previous_setpoint:.0f} to "
                         f"{value:.0f} kV at {ts:%H:%M:%S}; reactive power {moved} (about {reactive_before:.1f} "
                         f"to {reactive_after:.1f} MVAr), expected to {expected}.")
    else:
        story.append(f"Checks 2 and 3 not exercised: the voltage setpoint was held at {reference:.0f} kV "
                     f"and never stepped higher or lower in this capture, so the injection and "
                     f"absorption steps were not performed here.")
    if droop_steps:                                                            # CHECK 4
        for ts, value in droop_steps:
            story.append(f"Check 4 pass: the droop was changed to {value:.0f}% at {ts:%H:%M:%S}.")
    elif droop_col is not None:
        story.append(f"Check 4 not exercised: the droop stayed at {droop_held:.0f}% and the change to a "
                     f"second droop setting was not performed in this capture.")
    story.append(f"Across the window the plant held a small reactive trim near {median_reactive:.1f} MVAr "
                 f"to keep the voltage at its reference.")
    if setpoint_steps or droop_steps:
        story.append("Overall: voltage control is active, and the stepped parts of the test present in "
                     "this capture were assessed above.")
    else:
        story.append("Overall: voltage control is active and holding the POC voltage at its reference. "
                     "The stepped setpoint and droop-change parts of the test were not exercised in this "
                     "capture, so they are reported as not performed rather than passed.")

    print(f"\nVoltage mode window {n} of {total}  reference {reference:.0f} kV  figure {out_path.name}")
    for k, line in enumerate(story, start=1):
        print(f"  {k}. {line}")
    plt.show()


# 3. One graph and one set of findings per window.
voltage_windows = on_segments(v_region[v_mode_col])
print(f"Voltage-mode windows found: {len(voltage_windows)}")
if not voltage_windows:
    print("Voltage mode is never on in this capture, so the voltage test was not performed here.")
for n, (start, end) in enumerate(voltage_windows, start=1):
    plot_voltage_window(start, end, n, len(voltage_windows))

Voltage-mode windows found: 1



Voltage mode window 1 of 1  reference 137 kV  figure hartebeesthoek_voltage_080047.png
  1. Check 1 pass: while voltage mode was on the measured POC voltage held near 137.5 kV, within 0.5% of the 137 kV reference for 99% of the window.
  2. The droop was set to 4% for this window.
  3. Checks 2 and 3 not exercised: the voltage setpoint was held at 137 kV and never stepped higher or lower in this capture, so the injection and absorption steps were not performed here.
  4. Check 4 not exercised: the droop stayed at 4% and the change to a second droop setting was not performed in this capture.
  5. Across the window the plant held a small reactive trim near -3.9 MVAr to keep the voltage at its reference.
  6. Overall: voltage control is active and holding the POC voltage at its reference. The stepped setpoint and droop-change parts of the test were not exercised in this capture, so they are reported as not performed rather than passed.


C:\Users\erick\AppData\Local\Temp\ipykernel_22136\2607174273.py:216: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### What the voltage plot shows

Voltage mode was the one reactive-side control left on for the whole two hours, so this single graph covers the entire capture rather than a short window. The upper panel is the heart of the test: the measured voltage at the point of connection (blue) sits on its 137 kV setpoint (red dashed) from the first sample to the last, drifting only about a kilovolt either side. For 99 percent of the capture it stayed inside the grid code accuracy band of half a percent of nominal [2], so the plant held the reference it was given. The small dips and lifts that do appear line up with the shaded stretches, which are the active-power tests (curtailment, power gradient and delta) running underneath: as the plant's real power is pulled around, the local voltage moves a little and the controller trims it back.

The lower panel shows how that trimming is done. Reactive power stays small and mostly negative, hovering near -4 MVAr, so the plant is gently absorbing reactive power to keep the local voltage from drifting above its target. The two brief positive excursions, where it injects instead, again sit under the shaded active-power events, when a sudden change in real power briefly pushed the voltage the other way.

What this capture does not contain is the stepped part of the voltage test. The setpoint never moved off 137 kV and the droop stayed at 4 percent throughout, so the procedure steps that raise and lower the target and switch the droop to 8 percent, observing reactive power injected then absorbed at each [1], were not exercised here. They are reported as not performed rather than passed, because a step that never happened cannot be judged. The honest conclusion is a positive one as far as it goes: voltage control is active and holding the reference accurately, with the expected small reactive trim, and the natural next step is a capture that walks the setpoint and droop through their steps so the full mode can be assessed.

### Reactive power (Q) mode at available power

In reactive power mode the plant is commanded a reactive power setpoint in MVAr directly and must hold it. The procedure [1] sets the mode on with voltage and power factor modes off, then steps the setpoint through injection values (up to the plant's MVAr rating) and absorption values (down to the negative rating), checking the measured reactive power tracks each. This subsection finds every period where reactive power mode is on. If the mode was never switched on during the capture, that is reported plainly as not performed, since a test that did not run cannot be passed or failed.

In [28]:
# =============================================================================
# REACTIVE POWER (Q) MODE TEST  (at available power)
#
# What the plant must do, from the acceptance procedure [1] and grid code [2]:
#   CHECK 1  Q mode ON with voltage mode and power factor mode OFF -> reactive power
#            should follow the commanded MVAr setpoint.
#   CHECK 2  Setpoint stepped through injection and absorption values -> the measured
#            reactive power should track each within the grid-code accuracy.
#
# We find every period where Q mode is on. Where it is never on, the test was not
# performed in this capture and that is stated plainly, never faked.
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. The channels this test needs, found by role.
q_mode_col     = resolve("q_mode", required=True)   # reactive power mode on/off flag
reactive_col   = resolve("q_meas", required=True)   # measured reactive power (MVAr)
q_setpoint_col = resolve("q_sp")                     # reactive power setpoint (MVAr), if recorded

# 2. Restrict to the configured scope (whole record unless limited in the config cell).
q_cfg = EVENT_WINDOWS.get("reactive_power")
q_region = df.loc[q_cfg[0]:q_cfg[1]] if (q_cfg and all(q_cfg)) else df


def plot_q_window(mode_on, mode_off, n, total):
    win      = window_around(mode_on, mode_off)
    reactive = win[reactive_col]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(reactive.index, reactive, color="#9467bd", lw=1.6, label="reactive power (measured)")
    if q_setpoint_col is not None:
        setpoint = win[q_setpoint_col]
        ax.plot(setpoint.index, setpoint, color="#d62728", lw=1.8, ls="--", drawstyle="steps-post",
                label="reactive power setpoint")
        # value label on each commanded level, the same way the curtailment figure labels its ceilings
        for ts, val in setpoint[setpoint.ne(setpoint.shift())].items():
            ax.annotate(f"{val:.0f} MVAr", xy=(ts, val), xytext=(3, 4), textcoords="offset points",
                        ha="left", va="bottom", fontsize=7.0, color="#d62728",
                        bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.6))

    span = (reactive.max() - reactive.min()) or 1.0
    ax.set_ylim(reactive.min() - 0.15 * span, reactive.max() + 0.25 * span)
    events = [(mode_on, "#ff7f0e", f"{mode_on:%H:%M:%S}\nQ mode ON"),
              (mode_off, "#2ca02c", f"{mode_off:%H:%M:%S}\nQ mode OFF")]
    levels = mark_steps(ax, events)

    ax.set_xlabel(f"Time ({TIME_ZONE_LABEL})")
    ax.set_ylabel("Reactive power (MVAr)")
    ax.set_title(f"{SITE_NAME} reactive power (Q) mode window {n} of {total} at {mode_on:%H:%M:%S}",
                 pad=title_pad_for(levels))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.legend(loc="upper right", framealpha=0.9)
    out_path = OUTPUT_DIR / f"{SITE_SLUG}_reactive_power_{mode_on:%H%M%S}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    # --- Findings: did the measured reactive power track each commanded level? ---
    on_reactive = df[reactive_col].loc[mode_on:mode_off]
    story = []
    if q_setpoint_col is not None:
        on_setpoint = df[q_setpoint_col].loc[mode_on:mode_off]
        levels = constant_segments(on_setpoint)
        checks = 0
        for level_start, level_end, target in levels:
            tail_start = max(level_start, level_end - pd.Timedelta(seconds=REACTIVE_SETTLE_TAIL_SECONDS))
            measured = on_reactive.loc[tail_start:level_end]
            if not len(measured):
                continue
            settled = float(measured.median())
            tolerance = max(REACTIVE_POWER_ACCURACY_MVAR, 0.05 * abs(target))
            ok = abs(settled - target) <= tolerance
            checks += 1
            story.append(f"Check {checks} {'pass' if ok else 'fail'}: setpoint {target:.1f} MVAr from "
                         f"{level_start:%H:%M:%S}, measured reactive settled near {settled:.1f} MVAr "
                         f"({'within' if ok else 'outside'} the +/-{tolerance:.1f} MVAr accuracy).")
        if checks == 0:
            story.append(f"The reactive power setpoint could not be read while the mode was on, so the "
                         f"tracking cannot be judged; measured reactive ranged {on_reactive.min():.1f} to "
                         f"{on_reactive.max():.1f} MVAr.")
    else:
        story.append(f"No reactive power setpoint channel is recorded, so the measured reactive power "
                     f"cannot be judged against a target; it ranged {on_reactive.min():.1f} to "
                     f"{on_reactive.max():.1f} MVAr while the mode was on.")

    print(f"\nReactive power (Q) mode window {n} of {total}  figure {out_path.name}")
    for k, line in enumerate(story, start=1):
        print(f"  {k}. {line}")
    plt.show()


# 3. One graph per window, or an honest note when the mode was never on.
q_windows = on_segments(q_region[q_mode_col])
print(f"Reactive power (Q) mode windows found: {len(q_windows)}")
if not q_windows:
    print("Reactive power (Q) mode was not switched on at any point in this capture, so the "
          "Q-mode-at-available-power test was not performed and cannot be assessed from this "
          "record. It is reported here as not performed, neither passed nor failed.")
for n, (mode_on, mode_off) in enumerate(q_windows, start=1):
    plot_q_window(mode_on, mode_off, n, len(q_windows))

Reactive power (Q) mode windows found: 0
Reactive power (Q) mode was not switched on at any point in this capture, so the Q-mode-at-available-power test was not performed and cannot be assessed from this record. It is reported here as not performed, neither passed nor failed.


### Power factor (PF) mode

In power factor mode the plant holds a commanded power factor, supplying reactive power in proportion to its real output. The procedure [1] sets the mode on with voltage and reactive power modes off, then steps the setpoint through leading and lagging values (for example +0.975 down to -0.95), checking the measured power factor tracks each within the grid-code tolerance of plus or minus 0.02 [2]. This subsection finds every period where power factor mode is on. If the mode was never switched on during the capture, that is reported plainly as not performed.

In [29]:
# =============================================================================
# POWER FACTOR (PF) MODE TEST
#
# What the plant must do, from the acceptance procedure [1] and grid code [2]:
#   CHECK 1  PF mode ON with voltage mode and Q mode OFF -> the measured power factor
#            should follow the commanded setpoint.
#   CHECK 2  Setpoint stepped through leading and lagging values (for example +0.975 to
#            -0.95) -> the measured power factor should track each within +/-0.02.
#
# We find every period where PF mode is on. Where it is never on, the test was not
# performed in this capture and that is stated plainly, never faked.
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. The channels this test needs, found by role.
pf_mode_col     = resolve("pf_mode", required=True)  # power factor mode on/off flag
pf_col          = resolve("pf_meas", required=True)  # measured power factor
pf_setpoint_col = resolve("pf_sp")                    # power factor setpoint, if recorded
reactive_col    = resolve("q_meas")                   # measured reactive power, used to fix the sign at unity

# 2. Restrict to the configured scope (whole record unless limited in the config cell).
pf_cfg = EVENT_WINDOWS.get("power_factor")
pf_region = df.loc[pf_cfg[0]:pf_cfg[1]] if (pf_cfg and all(pf_cfg)) else df


def break_at_flips(series):
    # Power factor flips between +1 and -1 at unity (the same physical point), and a plain
    # line drawn across that jump shows as a false vertical bar. Lift the pen wherever the
    # value jumps by more than PF_PLOT_BREAK by setting that sample to "not a number", which
    # matplotlib leaves as a gap rather than a connecting line.
    broken = series.astype(float).copy()
    jumped = broken.diff().abs() > PF_PLOT_BREAK
    broken[jumped] = np.nan
    return broken


def display_power_factor(power_factor, reactive):
    # A readable measured trace. The leading or lagging sign is taken from the physical
    # reactive power, which is the unambiguous quantity; at unity, where reactive power is
    # about zero and the recorded power-factor sign is arbitrary, the trace is drawn at +1
    # (the same unity the setpoint uses), so it no longer paints a false line down at -1.
    if reactive is None:
        return break_at_flips(power_factor)
    signed = power_factor.abs() * np.sign(reactive)
    signed[reactive.abs() <= PF_UNITY_REACTIVE_MVAR] = 1.0
    return break_at_flips(signed)


def plot_pf_window(mode_on, mode_off, n, total):
    win          = window_around(mode_on, mode_off)
    power_factor = win[pf_col]
    reactive     = win[reactive_col] if reactive_col is not None else None

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(power_factor.index, display_power_factor(power_factor, reactive), color="#17becf",
            lw=1.6, label="power factor (measured)")
    if pf_setpoint_col is not None:
        setpoint = win[pf_setpoint_col]
        ax.plot(setpoint.index, setpoint, color="#d62728", lw=1.8, ls="--", drawstyle="steps-post",
                label="power factor setpoint")
        for ts, val in setpoint[setpoint.ne(setpoint.shift())].items():
            ax.annotate(f"{val:+.3f}", xy=(ts, val), xytext=(3, 4), textcoords="offset points",
                        ha="left", va="bottom", fontsize=7.0, color="#d62728",
                        bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.6))

    ax.set_ylim(-1.25, 1.25)            # power factor lives between -1 and 1; fixed limits keep it readable
    events = [(mode_on, "#ff7f0e", f"{mode_on:%H:%M:%S}\nPF mode ON"),
              (mode_off, "#2ca02c", f"{mode_off:%H:%M:%S}\nPF mode OFF")]
    levels = mark_steps(ax, events)

    ax.set_xlabel(f"Time ({TIME_ZONE_LABEL})")
    ax.set_ylabel("Power factor")
    ax.set_title(f"{SITE_NAME} power factor (PF) mode window {n} of {total} at {mode_on:%H:%M:%S}",
                 pad=title_pad_for(levels))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.legend(loc="upper right", framealpha=0.9)
    out_path = OUTPUT_DIR / f"{SITE_SLUG}_power_factor_{mode_on:%H%M%S}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    # --- Findings: did the measured power factor track each commanded level? ---
    # Power factor is judged by magnitude (how close to its commanded value), because the
    # sign flips between leading and lagging at unity and a single sample's sign is not
    # reliable there. The magnitude is steady through that flip, so it is the honest measure.
    on_pf = df[pf_col].loc[mode_on:mode_off]
    story = []
    if pf_setpoint_col is not None:
        on_setpoint = df[pf_setpoint_col].loc[mode_on:mode_off]
        levels = constant_segments(on_setpoint)
        checks = 0
        for level_start, level_end, target in levels:
            tail_start = max(level_start, level_end - pd.Timedelta(seconds=REACTIVE_SETTLE_TAIL_SECONDS))
            measured = on_pf.loc[tail_start:level_end].abs()
            if not len(measured):
                continue
            settled_magnitude = float(measured.median())
            target_magnitude = abs(target)
            ok = abs(settled_magnitude - target_magnitude) <= POWER_FACTOR_ACCURACY
            checks += 1
            story.append(f"Check {checks} {'pass' if ok else 'fail'}: setpoint {target:+.3f} from "
                         f"{level_start:%H:%M:%S}, measured power factor settled near {settled_magnitude:.3f} "
                         f"in magnitude ({'within' if ok else 'outside'} {POWER_FACTOR_ACCURACY:.2f} of the "
                         f"commanded {target_magnitude:.3f}).")
        if checks == 0:
            story.append("The power factor setpoint could not be read while the mode was on, so the "
                         "tracking cannot be judged in this window.")
        else:
            story.append("Leading or lagging (the sign) flips at unity, so the magnitude is judged here; "
                         "read the reactive power figure for the direction.")
    else:
        story.append("No power factor setpoint channel is recorded, so the measured power factor cannot "
                     "be judged against a target in this window.")

    print(f"\nPower factor (PF) mode window {n} of {total}  figure {out_path.name}")
    for k, line in enumerate(story, start=1):
        print(f"  {k}. {line}")
    plt.show()


# 3. One graph per window, or an honest note when the mode was never on.
pf_windows = on_segments(pf_region[pf_mode_col])
print(f"Power factor (PF) mode windows found: {len(pf_windows)}")
if not pf_windows:
    print("Power factor (PF) mode was not switched on at any point in this capture, so the "
          "PF-mode test was not performed and cannot be assessed from this record. It is "
          "reported here as not performed, neither passed nor failed.")
for n, (mode_on, mode_off) in enumerate(pf_windows, start=1):
    plot_pf_window(mode_on, mode_off, n, len(pf_windows))

Power factor (PF) mode windows found: 0
Power factor (PF) mode was not switched on at any point in this capture, so the PF-mode test was not performed and cannot be assessed from this record. It is reported here as not performed, neither passed nor failed.


## Stop and start

The stop and start test is the most absolute in the procedure [1]: on a stop signal the plant must ramp its active power all the way down to zero, and on a start signal it must ramp back up to its running level. There is no setpoint to track and no mode flag for it in the logger, so the test is read straight from the active power trace, in the same spirit as the frequency response: a fall to zero is the stop, the flat run at zero is the plant held off, and the climb back up is the start.

This section finds every stop and start cycle in the record on its own, treating active power below a small fraction of the plant's own running maximum as stopped, and draws one graph per cycle. It marks the moment the output reaches zero and the moment it starts climbing back, and reports how far down it went and what level it returned to.

In [30]:
# =============================================================================
# STOP AND START TEST
#
# What the plant must do, from the acceptance procedure [1]:
#   CHECK 1  Stop signal  -> the plant ramps its active power down to zero.
#   CHECK 2  Start signal -> the plant ramps back up to its running level.
#
# There is no stop/start mode flag, so each cycle is read from the active power itself: a
# stretch where the output sits near zero is the plant stopped, the fall into it is the
# stop and the climb out of it is the start. We find every such cycle and draw one graph
# and one set of findings per cycle. Every value is read from the data.
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. The channel this test needs, found by role (never by a fixed column name).
power_col = resolve("poc_p", required=True)   # measured active power

# 2. Restrict to the configured scope (whole record unless limited in the config cell).
ss_cfg = EVENT_WINDOWS.get("stop_start")
ss_region = df.loc[ss_cfg[0]:ss_cfg[1]] if (ss_cfg and all(ss_cfg)) else df
power_all = ss_region[power_col]

# 3. Work out what "stopped" means for this plant, straight from the data.
running_maximum = float(power_all.max())
stop_level = STOP_FRACTION * running_maximum    # at or below this counts as stopped
is_stopped = power_all <= stop_level


def level_before(moment, lookback_seconds=STOP_LEVEL_BEFORE_SECONDS):
    """The active power level the plant was running at just before a moment, taken as the
    median of the samples in the lookback window that are above the stop level."""
    before = power_all.loc[moment - pd.Timedelta(seconds=lookback_seconds):moment]
    running = before[before > stop_level]
    return float(running.median()) if len(running) else float("nan")


def level_after(moment, lookahead_seconds=STOP_LEVEL_AFTER_SECONDS):
    """The active power level the plant climbed back to just after a moment, taken as the
    median of the samples in the lookahead window that are above the stop level."""
    after = power_all.loc[moment:moment + pd.Timedelta(seconds=lookahead_seconds)]
    running = after[after > stop_level]
    return float(running.median()) if len(running) else float("nan")


def plot_stop_start(stop_begin, start_end, n, total):
    win   = window_around(stop_begin, start_end, before="120s", after="150s")
    power = win[power_col]

    before_level = level_before(stop_begin)                          # level it ran at before
    bottom_level = float(power_all.loc[stop_begin:start_end].min())  # lowest while stopped
    after_level  = level_after(start_end)                            # level it returned to

    # --- The acceptance checks, computed plainly from the measured data ------
    CHECK1_stopped   = bottom_level <= stop_level
    CHECK2_restarted = (pd.notna(after_level) and pd.notna(before_level)
                        and after_level >= STOP_RECOVERY_FRACTION * before_level)

    # --- Draw the graph ------------------------------------------------------
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(power.index, power, color="#1f77b4", lw=1.6, label="active power (measured)")
    if pd.notna(before_level):
        ax.axhline(before_level, color="#7f7f7f", ls="--", lw=1.2,
                   label=f"level before stop {before_level:.0f} MW")
    ax.axhline(0.0, color="#bbbbbb", ls="-", lw=1.0)

    events = [(stop_begin, "#d62728", f"{stop_begin:%H:%M:%S}\nstopped, power at zero"),
              (start_end, "#2ca02c", f"{start_end:%H:%M:%S}\nstart, ramping back up")]
    top = max(power.max(), before_level if pd.notna(before_level) else power.max())
    ax.set_ylim(min(0.0, power.min()) - 4, top + 22)
    mark_events(ax, events, top, gap=6, fontsize=8.5)

    ax.set_xlabel(f"Time ({TIME_ZONE_LABEL})")
    ax.set_ylabel("Active power (MW)")
    ax.set_title(f"{SITE_NAME} stop and start cycle {n} of {total} at {stop_begin:%H:%M:%S}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.legend(loc="upper right", framealpha=0.9)
    out_path = OUTPUT_DIR / f"{SITE_SLUG}_stop_start_{stop_begin:%H%M%S}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    # --- Findings, written straight from the checks above --------------------
    story = []
    if CHECK1_stopped:
        story.append(f"Check 1 pass: the output fell from about {before_level:.0f} MW down to zero "
                     f"by {stop_begin:%H:%M:%S}, so the plant ramped down on the stop signal.")
    else:
        story.append(f"Check 1 fail: the lowest the output reached was about {bottom_level:.0f} MW, above the "
                     f"{stop_level:.0f} MW stop level, so it did not ramp fully to zero here.")
    if CHECK2_restarted:
        story.append(f"Check 2 pass: after {start_end:%H:%M:%S} the output climbed back to about {after_level:.0f} MW, "
                     f"near its {before_level:.0f} MW level before the stop, so the plant ramped back up on the start signal.")
    elif pd.isna(after_level):
        story.append(f"The start at {start_end:%H:%M:%S} sits at the end of the record, so the climb back is not captured here.")
    else:
        story.append(f"Check 2: after {start_end:%H:%M:%S} the output recovered to about {after_level:.0f} MW, below its "
                     f"{before_level:.0f} MW level before the stop, so the climb back is only partial in this window.")
    stopped_minutes = (start_end - stop_begin).total_seconds() / 60.0
    story.append(f"The plant was held at zero for about {stopped_minutes:.0f} minutes between the stop and the start.")

    print(f"\nStop and start cycle {n} of {total}  figure {out_path.name}")
    for k, line in enumerate(story, start=1):
        print(f"  {k}. {line}")
    plt.show()


# 4. One graph and one set of findings per cycle.
stop_windows = on_segments(is_stopped)
print(f"Running maximum read from the data: {running_maximum:.0f} MW; stop level {stop_level:.1f} MW")
print(f"Stop and start cycles found: {len(stop_windows)}")
if not stop_windows:
    print("The active power never falls to zero in this record, so no stop and start cycle was performed here.")
for n, (stop_begin, start_end) in enumerate(stop_windows, start=1):
    plot_stop_start(stop_begin, start_end, n, len(stop_windows))

Running maximum read from the data: 77 MW; stop level 3.9 MW
Stop and start cycles found: 2



Stop and start cycle 1 of 2  figure hartebeesthoek_stop_start_084025.png
  1. Check 1 pass: the output fell from about 48 MW down to zero by 08:40:25, so the plant ramped down on the stop signal.
  2. Check 2 pass: after 08:44:06 the output climbed back to about 47 MW, near its 48 MW level before the stop, so the plant ramped back up on the start signal.
  3. The plant was held at zero for about 4 minutes between the stop and the start.


C:\Users\erick\AppData\Local\Temp\ipykernel_22136\1695064030.py:103: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Stop and start cycle 2 of 2  figure hartebeesthoek_stop_start_090919.png
  1. Check 1 pass: the output fell from about 51 MW down to zero by 09:09:19, so the plant ramped down on the stop signal.
  2. Check 2 pass: after 09:13:01 the output climbed back to about 56 MW, near its 51 MW level before the stop, so the plant ramped back up on the start signal.
  3. The plant was held at zero for about 4 minutes between the stop and the start.


C:\Users\erick\AppData\Local\Temp\ipykernel_22136\1695064030.py:103: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### What the stop and start plots show

Both cycles are textbook. In each one the output is running steadily, near 48 megawatts (MW) in the first cycle and 51 MW in the second, when the stop signal lands and the power falls almost vertically to zero, reaching it within about a quarter of a minute. It then sits flat at zero for roughly four minutes, the plant held off, before the start signal sends it climbing back up just as steeply to the level it had been running at, with a small overshoot at the top of the first cycle before it settles.

There is nothing ambiguous to weigh here: the plant did exactly what the stop and start test asks, ramping fully down on the stop and fully back up on the start, on both occasions. The two cycles also frame the active-power story of the rest of the capture, since the curtailment, power gradient and delta tests all sit in the running stretches between these stops.

## AGC signal verification

AGC stands for Automatic Generation Control, the system by which the grid operator nudges a plant's output up and down second by second to keep the whole grid balanced. The verification in the procedure [1] is mostly a telemetry audit: it lists the AGC signals that must be telemetered rather than estimated, the high and low regulating limits and the regulation range between them, the ramp rate, the sent-out and generated values, the AGC status, and the setpoint feedback, and asks for each whether it is present and whether it changes. A second part asks for the plant to be moved by at least 20 to 30 megawatts (MW) through pulses or setpoints so the control system's response can be observed.

This section looks for those AGC signals in the record by their role. Where they are present it reports their value and whether it changes; where they are not, it says so plainly rather than inventing them, because the audit cannot be passed on signals that were never logged. It then checks the observable part, whether the active power was actually moved by at least the required amount during the capture.

In [31]:
# =============================================================================
# AGC SIGNAL VERIFICATION
#
# What the procedure asks for [1]:
#   CHECK 1  The AGC telemetry signals (high and low regulating limits, regulation range,
#            ramp rate, sent-out value, generated value, AGC status, setpoint feedback) are
#            present and telemetered, not estimated.
#   CHECK 2  The plant is moved by at least AGC_MOVE_MW through pulses or setpoints, so its
#            control response can be observed.
#
# We resolve each AGC signal by role. Signals that are not in this record are reported as
# not captured, never invented. We then check the observable part, whether the active power
# moved by at least the required amount anywhere in the record.
# =============================================================================

# 1. The AGC signals the procedure asks to verify, each found by role.
agc_signals = {
    "high regulating limit": resolve("hi_limit"),
    "low regulating limit":  resolve("lo_limit"),
    "ramp rate up":          resolve("ramp_up"),
    "ramp rate down":        resolve("ramp_down"),
    "sent-out value":        resolve("sentout"),
    "generated value":       resolve("generated"),
    "AGC status":            resolve("agc_mode"),
    "setpoint feedback":     resolve("sp_feedback"),
}
power_col = resolve("poc_p", required=True)   # measured active power

# 2. CHECK 1: which AGC signals are actually telemetered in this record?
present = {name: col for name, col in agc_signals.items() if col is not None}
absent  = [name for name, col in agc_signals.items() if col is None]

print("AGC signal verification")
print(f"  Signals present in this record: {len(present)} of {len(agc_signals)}")
for name, col in present.items():
    series  = pd.to_numeric(df[col], errors="coerce")
    changes = int(series.ne(series.shift()).sum()) - 1
    current = float(series.dropna().iloc[-1]) if series.notna().any() else float("nan")
    movement = "changes over the record" if changes > 0 else "holds steady"
    print(f"    {name}: current value {current:.2f}, {movement}")
if absent:
    print("  Signals not captured in this record (cannot be verified, reported as not telemetered):")
    for name in absent:
        print(f"    {name}")

# 3. CHECK 2: was the plant moved by at least the required amount?
power = pd.to_numeric(df[power_col], errors="coerce")
power_range  = float(power.max() - power.min())
moved_enough = power_range >= AGC_MOVE_MW
print(f"\n  Plant movement observed: active power ranged over {power_range:.0f} MW "
      f"(from {power.min():.0f} MW to {power.max():.0f} MW).")
if moved_enough:
    print(f"  Check 2 pass: the plant was moved by at least the {AGC_MOVE_MW:.0f} MW the procedure asks for, "
          f"so its response to setpoint and mode commands is observable in the active-power tests above.")
else:
    print(f"  Check 2: the plant moved less than the {AGC_MOVE_MW:.0f} MW the procedure asks for in this record.")

# 4. Honest overall verdict.
if absent:
    print("\n  Overall: the AGC-specific telemetry (regulating limits, sent-out and generated values, AGC "
          "status and setpoint feedback) is not logged in this record, so the AGC signal verification table "
          "cannot be completed from this capture. It is reported as not performed rather than passed or "
          "failed. The plant movement the second part asks for is present, and the active-power sections "
          "above show how the plant follows setpoint and mode commands.")
else:
    print("\n  Overall: all AGC signals the procedure lists are present and were reported above.")

AGC signal verification
  Signals present in this record: 2 of 8
    ramp rate up: current value 5.00, changes over the record
    ramp rate down: current value 10.00, changes over the record
  Signals not captured in this record (cannot be verified, reported as not telemetered):
    high regulating limit
    low regulating limit
    sent-out value
    generated value
    AGC status
    setpoint feedback

  Plant movement observed: active power ranged over 78 MW (from -1 MW to 77 MW).
  Check 2 pass: the plant was moved by at least the 20 MW the procedure asks for, so its response to setpoint and mode commands is observable in the active-power tests above.

  Overall: the AGC-specific telemetry (regulating limits, sent-out and generated values, AGC status and setpoint feedback) is not logged in this record, so the AGC signal verification table cannot be completed from this capture. It is reported as not performed rather than passed or failed. The plant movement the second part asks for

## Observations

This notebook worked through the SCADA functionality test record one test at a time, reading every value from the logger and judging it against the acceptance procedure [1] and, where a grid-code curve or limit applied, the South African Grid Code requirements for renewable power plants [2]. Across the whole capture the plant's control functions are present and largely correct, with a small number of clear items to follow up.

The active-power constraint tests pass cleanly. Curtailment pulled the output down to each commanded ceiling and released it again when the mode was switched off. The power gradient limiter held the ramp close to its commanded rate in both the up and the down directions. The stop and start test was the most clear-cut of all: on each of two cycles the output ramped fully to zero on the stop signal and back to the level it had been running at on the start signal.

The reserve and reactive-side results pass with detail worth carrying forward. The delta production constraint held its reserve accurately in the 15 percent window, settling right on the target, but under-delivered in the 10 percent window, where it touched the target briefly and then drifted back up, so the reserve was established and then partly given back. Voltage control held the point of connection voltage on its 137 kilovolt reference inside the grid-code accuracy band for almost the entire capture, trimming a small amount of reactive power to do so.

Two findings are constructive rather than clean passes. The frequency response is present and in the right direction, cutting active power hard above the over-frequency point and recovering once the frequency drops back, and it correctly stayed connected because the real grid frequency never approached the trip point. But it over-reduces against the Figure 6 droop curve, behaving like an on-off curtailment instead of a proportional droop, so the recommended next step is to retune the droop so the reduction tracks the curve.

Finally, several tests were simply not exercised in this capture, and are reported as such rather than assumed. The stepped part of the voltage test and the change of droop never occurred, the reactive power (Q) mode and the power factor (PF) mode were never switched on, and the AGC signal verification could not be completed because most of its telemetry is not logged in this record. Read together, the plant demonstrates working curtailment, power gradient, delta, voltage control and stop and start, a frequency response that needs a droop retune, and a group of reactive and AGC checks that need a follow-up capture in which those modes and signals are actually exercised and telemetered.

## References

[1] National Control System Support (NCSS), "SCADA Functionality Test Record," National Transmission Company South Africa SOC Ltd, Revision 3, 2026.

[2] "Grid Connection Code for Renewable Power Plants (RPPs) Connected to the Electricity Transmission System (TS) or the Distribution System (DS) in South Africa," Version 3.1, 2022.